In [1]:
#Cellpose auf bilder laufen lassen. Setze die directories und den Marker Namen

# 📦 Standard-Imports
import os, gc, torch, geojson
import numpy as np
import pandas as pd
from pathlib import Path
from tifffile import imread
from PIL import Image
from datetime import datetime
from cellpose.models import CellposeModel
from cellpose import io as cp_io
from skimage.measure import regionprops, label
from shapely.geometry import Polygon, mapping
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import seaborn as sns
import imageio.v2 as imageio
from time import time

Image.MAX_IMAGE_PIXELS = None

# 📍 Marker festlegen (dieser Code ist spezifisch für Vim)
marker = "Cxcl13" # z.B. "Vim", "DAPI", "CD45", "BrdU", "Cd52", "Cxcl13"

# 🔢 Kanalzuordnung
channel_map = {
    "DAPI": 0,
    "BrdU": 1,
    "CD45": 1,
    "Cd52": 2,
    "Cxcl13": 2,
    "Vim": 3
}

# 📂 Input/Output/Model
input_dir = Path(r"C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\testing_cellpose_modular\input_images\Cxcl13_images")
output_dir = Path(r"C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\testing_cellpose_modular\outputs\Cxcl13_testing")
model_dir = Path(r"C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\testing_cellpose_modular\models")

# 📋 Alle Bilder im Ordner laden
filenames = list(input_dir.glob("*.ome.tif"))

def get_model_for_marker(marker: str, model_dir: Path):
    """
    Wähle das passende Modell für einen Marker aus dem Model-Ordner.
    Für DAPI wird das eingebaute nuclei-Modell verwendet.
    """
    if marker == "DAPI":
        print("🧠 Verwende eingebautes Cellpose-Modell für DAPI (nuclei)")
        return CellposeModel(gpu=torch.cuda.is_available(), model_type="nuclei")

    # Suche Modelldatei im Ordner, die den Marker im Namen enthält
    model_files = list(model_dir.glob("*"))
    matched = [f for f in model_files if marker.lower() in f.name.lower()]

    if not matched:
        raise FileNotFoundError(f"❌ Kein Modell gefunden für Marker: {marker} in {model_dir}")
    if len(matched) > 1:
        print(f"⚠️ Mehrere Modelle gefunden, nehme das erste: {matched[0].name}")

    print(f"✅ Verwende Modell für {marker}: {matched[0].name}")
    return CellposeModel(pretrained_model=str(matched[0]), gpu=torch.cuda.is_available())

def run_cellpose_and_save_raw(image_path: Path):
    image_stem = image_path.name.replace(".ome.tif", "").replace(".tif", "")
    print(f"\n📂 Bild: {image_stem}")

    # 🛑 Skip, wenn RAW-GeoJSON bereits existiert
    marker_upper = marker.upper()
    parts = image_stem.split("_")
    sample_id = parts[0] + "_" + parts[1] if len(parts) >= 2 else image_stem  # e.g., C6_33
    save_dir = output_dir / sample_id / f"{sample_id}__{marker_upper}"
    raw_geojson_path = save_dir / f"{sample_id}__{marker_upper}_raw.geojson"
    if raw_geojson_path.exists():
        print(f"⏭️  Überspringe {sample_id}/{marker_upper}: {raw_geojson_path.name} existiert bereits.")
        return

    t_start = time()
    img = None
    input_img = None
    masks = None
    model = None

    try:
        # 🔢 Bild laden
        img = cp_io.imread(str(image_path))
        if img.ndim == 3 and img.shape[-1] < img.shape[0]:
            img = np.transpose(img, (2, 0, 1))  # (C, H, W)

        # 🧠 Kanäle festlegen
        marker_channel = channel_map[marker]
        channels = [marker_channel] if marker == "DAPI" else [marker_channel, channel_map["DAPI"]]
        input_img = img[channels, :, :]

        # 🧠 Modell laden
        model = get_model_for_marker(marker, model_dir)
        print(f"🔍 Modell: {'builtin (nuclei)' if marker == 'DAPI' else model.pretrained_model}")

        # 📁 Output-Ordner
        save_dir.mkdir(parents=True, exist_ok=True)

        # 🚀 Cellpose ausführen
        masks, _, _ = model.eval(
            input_img,
            diameter=30,
            normalize=True,
            flow_threshold=0.4,
            cellprob_threshold=0.0,
            progress=True
        )

        # 🧼 Maske speichern
        Image.fromarray((masks > 0).astype(np.uint8) * 255).save(save_dir / f"{sample_id}__{marker_upper}_mask_raw.png")
        np.save(save_dir / f"{sample_id}__{marker_upper}_masks.npy", masks)

        # 📊 Regionprops berechnen
        labeled = label(masks)
        regions_marker = regionprops(labeled, intensity_image=img[marker_channel])
        regions_dapi = regionprops(labeled, intensity_image=img[channel_map["DAPI"]])

        props = [{
            "cell_id": int(r.label),
            "area": int(r.area),
            f"mean_intensity_{marker.lower()}": float(r.mean_intensity),
            "mean_intensity_dapi": float(r_dapi.mean_intensity)
        } for r, r_dapi in zip(regions_marker, regions_dapi)]

        df = pd.DataFrame(props)

        # 📁 Speichere direkt in Excel-Datei mit Marker-Tab
        excel_path = save_dir / f"{sample_id}__{marker_upper}_intensity_data.xlsx"
        sheet_name = marker

        if excel_path.exists():
            with pd.ExcelWriter(excel_path, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            with pd.ExcelWriter(excel_path, mode="w", engine="openpyxl") as writer:
                df.to_excel(writer, sheet_name=sheet_name, index=False)

        # 🌍 GeoJSON (raw)
        def to_geo(r):
            try:
                poly = Polygon([(int(c[1]), int(c[0])) for c in r.coords]).convex_hull
                return geojson.Feature(geometry=mapping(poly), properties={
                    "name": marker,
                    "classification": f"{marker}_raw",
                    "cell_id": int(r.label)
                })
            except Exception:
                return None

        features = Parallel(n_jobs=-1)(delayed(to_geo)(r) for r in regions_marker)
        features = [f for f in features if f is not None]

        with open(save_dir / f"{sample_id}__{marker_upper}_raw.geojson", 'w') as f:
            geojson.dump(geojson.FeatureCollection(features), f)

        # 📉 Histogramm (roh)
        plt.figure(figsize=(8, 5))
        sns.histplot(df[f"mean_intensity_{marker.lower()}"], bins=255, binrange=(0, 255),
                     color="darkorange", edgecolor="black")
        plt.title(f"{marker} Intensity Histogram")
        plt.tight_layout()
        plt.savefig(save_dir / f"{sample_id}__{marker_upper}_histogram_intensity.png", dpi=300)
        plt.close()

        # 🟢 Scatterplot (raw)
        plt.figure(figsize=(6, 6))
        sns.scatterplot(data=df,
                        x="mean_intensity_dapi",
                        y=f"mean_intensity_{marker.lower()}",
                        edgecolor=None, alpha=0.5)
        plt.xlabel("Mean DAPI Intensity")
        plt.ylabel(f"Mean {marker} Intensity")
        plt.title(f"{marker} vs DAPI")
        plt.tight_layout()
        plt.savefig(save_dir / f"{sample_id}__{marker_upper}_scatter_dapi_vs_intensity.png", dpi=300)
        plt.close()

    finally:
        # sicheres Cleanup
        try:
            del img, input_img, masks, model
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        t_end = time()
        duration = t_end - t_start
        print(f"✅ Fertig in {duration:.1f} Sekunden")

# 📂 Cellpose auf alle Bilder laufen lassen (mit Anzeige, was geskippt wird)
for path in filenames:
    print(f"▶️  Check {path.name} …")
    run_cellpose_and_save_raw(path)

# 🧹 Speicher aufräumen – nur die wichtigsten Variablen behalten
keep_vars = {
    "run_cellpose_and_save_raw",
    "filenames",
    "marker",
    "channel_map",
    "input_dir",
    "output_dir",
    "model_dir",
    "gc",
    "torch",
}

for var in list(globals().keys()):
    if var not in keep_vars and not var.startswith("__") and var not in dir(__builtins__):
        try:
            del globals()[var]
        except Exception:
            pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("🧽 Speicher aufgeräumt. Nur benötigte Variablen wurden behalten.")
print("🎉 Hey, endlich ganz fertig!")




Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.18 
torch version:  	2.5.1+cu121! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


▶️  Check C11_33_CD45-647_Cxcl13-568_Vim-488.ome.tif …

📂 Bild: C11_33_CD45-647_Cxcl13-568_Vim-488
✅ Verwende Modell für Cxcl13: Cxcl13_20250814_1
🔍 Modell: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\testing_cellpose_modular\models\Cxcl13_20250814_1


✅ Fertig in 800.2 Sekunden
▶️  Check C7_5_CD45-647_Cxcl13-568_Vim-488.ome.tif …

📂 Bild: C7_5_CD45-647_Cxcl13-568_Vim-488
✅ Verwende Modell für Cxcl13: Cxcl13_20250814_1
🔍 Modell: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\testing_cellpose_modular\models\Cxcl13_20250814_1


✅ Fertig in 378.1 Sekunden
▶️  Check C9_32_CD45-647_Cxcl13-568_Vim-488.ome.tif …

📂 Bild: C9_32_CD45-647_Cxcl13-568_Vim-488
✅ Verwende Modell für Cxcl13: Cxcl13_20250814_1
🔍 Modell: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\testing_cellpose_modular\models\Cxcl13_20250814_1


✅ Fertig in 625.1 Sekunden
▶️  Check I10_35_CD45-647_Cxcl13-568_Vim-488.ome.tif …

📂 Bild: I10_35_CD45-647_Cxcl13-568_Vim-488
✅ Verwende Modell für Cxcl13: Cxcl13_20250814_1
🔍 Modell: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\testing_cellpose_modular\models\Cxcl13_20250814_1


✅ Fertig in 595.7 Sekunden
▶️  Check I4_18_CD45-647_Cxcl13-568_Vim-488.ome.tif …

📂 Bild: I4_18_CD45-647_Cxcl13-568_Vim-488
✅ Verwende Modell für Cxcl13: Cxcl13_20250814_1
🔍 Modell: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\testing_cellpose_modular\models\Cxcl13_20250814_1


✅ Fertig in 426.3 Sekunden
▶️  Check I6_25_CD45-647_Cxcl13-568_Vim-488.ome.tif …

📂 Bild: I6_25_CD45-647_Cxcl13-568_Vim-488
✅ Verwende Modell für Cxcl13: Cxcl13_20250814_1
🔍 Modell: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\testing_cellpose_modular\models\Cxcl13_20250814_1
✅ Fertig in 275.8 Sekunden
▶️  Check I7_33_CD45-647_Cxcl13-568_Vim-488.ome.tif …

📂 Bild: I7_33_CD45-647_Cxcl13-568_Vim-488
✅ Verwende Modell für Cxcl13: Cxcl13_20250814_1
🔍 Modell: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\testing_cellpose_modular\models\Cxcl13_20250814_1


✅ Fertig in 497.6 Sekunden
▶️  Check I9_35_CD45-647_Cxcl13-568_Vim-488.ome.tif …

📂 Bild: I9_35_CD45-647_Cxcl13-568_Vim-488
✅ Verwende Modell für Cxcl13: Cxcl13_20250814_1
🔍 Modell: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\testing_cellpose_modular\models\Cxcl13_20250814_1


✅ Fertig in 371.6 Sekunden
🧽 Speicher aufgeräumt. Nur benötigte Variablen wurden behalten.
🎉 Hey, endlich ganz fertig!


In [2]:
#manual annotation; code to create graphs and everything just using my manually created geojson
# === Manuelle GeoJSONs (inkl. MultiPoint) → identische Outputs wie Cellpose ===
# 📦 Imports
import os, gc, geojson, warnings, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from pathlib import Path
from tifffile import imread as tiff_imread, TiffFile
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import shape, Polygon, MultiPolygon, Point, MultiPoint, mapping
from shapely import affinity
from skimage.draw import polygon as sk_poly, ellipse as sk_ellipse
from skimage.measure import regionprops, label

Image.MAX_IMAGE_PIXELS = None
warnings.filterwarnings("ignore", category=UserWarning)

# ===== Marker & Kanäle =====
marker = "Cxcl13"  # z.B. "Vim", "DAPI", "CD45", "BrdU", "Cd52", "Cxcl13"
channel_map = {
    "DAPI": 0,
    "BrdU": 1,
    "CD45": 1,
    "Cd52": 2,
    "Cxcl13": 2,
    "Vim": 3
}

# ===== Einstellungen =====
CIRCLE_DIAMETER_UM = 7.0     # deine Kreise in QuPath ≈ 7 µm
HIST_BINS = 255
HIST_RANGE = (0, 255)        # falls 16-bit: auf (0, 65535) ändern

# ===== Pfade =====
# Basis der manuellen GeoJSONs
base_manual_dir = Path(r"C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cxcl13_images_analyzed")

# Wo liegen die OME-TIFs?
export_roots = [
    Path(r"D:\Image_analysis\QuPath_project_overall\export_sections\CD45-647_Cxcl13-568_Vim-488"),
    Path(r"D:\Image_analysis\QuPath_project_overall\export_sections\BrdU-647_Vim-488"),
    Path(r"D:\Image_analysis\QuPath_project_overall\export_sections\CD45-647_CD52-568_Vim-488"),
]

# Ziel wie im Cellpose-Script
output_dir = Path(r"C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cxcl13_images_analyzed")

# ===== Utils =====
def log(msg: str):
    from time import strftime
    print(f"[{strftime('%H:%M:%S')}] {msg}")

def find_image_for_sample(sample_id: str) -> Path:
    """Suche passendes OME-TIF; bevorzuge Ordner mit 'Cxcl13' im Pfad."""
    cands = []
    for root in export_roots:
        cands.extend(list(root.glob(f"{sample_id}_*.ome.tif")))
    if not cands:
        raise FileNotFoundError(f"Kein OME-TIF für Sample {sample_id} unter export_roots gefunden.")
    pref = [p for p in cands if "cxcl13" in str(p.parent).lower()]
    return pref[0] if pref else cands[0]

def load_channels(path: Path, chan_dapi: int, chan_marker: int) -> tuple[np.ndarray, np.ndarray]:
    """Lädt DAPI & Marker als 2D Arrays; unterstützt (C,H,W) und (H,W,C)."""
    img = tiff_imread(str(path))
    if img.ndim != 3:
        raise ValueError(f"Unerwartete Bildform: {img.shape} in {path}")
    if img.shape[0] in (3,4,5,6):   # (C,H,W)
        dapi = img[chan_dapi].astype(np.float32)
        mker = img[chan_marker].astype(np.float32)
    elif img.shape[-1] in (3,4,5,6):  # (H,W,C)
        dapi = img[..., chan_dapi].astype(np.float32)
        mker = img[..., chan_marker].astype(np.float32)
    else:
        dapi = img[chan_dapi].astype(np.float32)
        mker = img[chan_marker].astype(np.float32)
    return dapi, mker

def get_mpp_from_ome(tif_path: Path, fallback=0.249) -> tuple[float, float]:
    """Liefert (mpp_x, mpp_y) in µm/px aus OME-XML; Fallback wenn nicht vorhanden."""
    mpp_x = mpp_y = fallback
    try:
        with TiffFile(str(tif_path)) as tf:
            if tf.ome_metadata:
                root = ET.fromstring(tf.ome_metadata)
                ns = {"ome": "http://www.openmicroscopy.org/Schemas/OME/2016-06"}
                px = root.find(".//ome:Pixels", ns)
                if px is not None:
                    if "PhysicalSizeX" in px.attrib:
                        mpp_x = float(px.attrib["PhysicalSizeX"])
                    if "PhysicalSizeY" in px.attrib:
                        mpp_y = float(px.attrib["PhysicalSizeY"])
    except Exception:
        pass
    return mpp_x, mpp_y

def geometry_to_mask(geom, H: int, W: int, rx_px: float = None, ry_px: float = None) -> np.ndarray:
    """
    Rastert Polygon/MultiPolygon/Point/MultiPoint nach Maske (True in ROI).
    Points/MultiPoints werden als Ellipsen mit Radien rx_px/ry_px (in Pixel) interpretiert.
    GeoJSON-Koordinaten sind [x,y]; skimage erwartet (r=y, c=x).
    """
    mask = np.zeros((H, W), dtype=bool)

    def fill_polygon(poly: Polygon):
        if poly.is_empty:
            return
        xs, ys = zip(*list(poly.exterior.coords))
        rr, cc = sk_poly(np.asarray(ys, float), np.asarray(xs, float), shape=(H, W))
        mask[rr, cc] = True
        for interior in poly.interiors:
            xs_h, ys_h = zip(*list(interior.coords))
            rr_h, cc_h = sk_poly(np.asarray(ys_h, float), np.asarray(xs_h, float), shape=(H, W))
            mask[rr_h, cc_h] = False

    if isinstance(geom, Polygon):
        fill_polygon(geom)
    elif isinstance(geom, MultiPolygon):
        for p in geom.geoms:
            fill_polygon(p)
    elif isinstance(geom, Point):
        rrad = int(round(ry_px)) if ry_px else 14
        crad = int(round(rx_px)) if rx_px else 14
        rr, cc = sk_ellipse(float(geom.y), float(geom.x), rrad, crad, shape=(H, W))
        mask[rr, cc] = True
    elif isinstance(geom, MultiPoint):
        rrad = int(round(ry_px)) if ry_px else 14
        crad = int(round(rx_px)) if rx_px else 14
        for pt in geom.geoms:
            rr, cc = sk_ellipse(float(pt.y), float(pt.x), rrad, crad, shape=(H, W))
            mask[rr, cc] = True

    return mask

def point_to_ellipse_polygon(x: float, y: float, rx: float, ry: float, resolution: int = 16) -> Polygon:
    """
    Erzeugt ein Ellipsen-Polygon (Shapely) um (x,y) mit Radien rx/ry in Pixeln.
    Wird benutzt, um im *_raw.geojson* aus Punkten echte Flächen zu schreiben.
    """
    circ = Point(0.0, 0.0).buffer(1.0, resolution=resolution)  # Einheitskreis
    ell  = affinity.scale(circ, xfact=rx, yfact=ry)
    ell  = affinity.translate(ell, xoff=x, yoff=y)
    return ell

def summarize_feature_mask(mask: np.ndarray, dapi: np.ndarray, mker: np.ndarray, cell_id: int) -> dict:
    if mask.sum() == 0:
        return {
            "cell_id": cell_id,
            "area": 0,
            f"mean_intensity_{marker.lower()}": 0.0,
            "mean_intensity_dapi": 0.0
        }
    return {
        "cell_id": cell_id,
        "area": int(mask.sum()),
        f"mean_intensity_{marker.lower()}": float(mker[mask].mean()),
        "mean_intensity_dapi": float(dapi[mask].mean())
    }

# ===== Hauptfunktion =====
def process_manual_geojson(manual_geojson_path: Path):
    """
    Liest manuelles GeoJSON (Polygon/MultiPolygon/Point/MultiPoint), holt Bild & mpp,
    rastert ggf. Kreise (7 µm) → erzeugt identische Outputs wie dein Cellpose-Script.
    """
    # z.B. ...\C10_17\C10_17__Cxcl13\C10_17__Cxcl13_manual.geojson
    sample_id = manual_geojson_path.stem.replace(f"__{marker}_manual", "")
    parts = sample_id.split("_")
    sample_id_norm = parts[0] + "_" + parts[1] if len(parts) >= 2 else sample_id
    marker_upper = marker.upper()

    # === Zielordner exakt wie im Cellpose-Script ===
    save_dir = output_dir / sample_id_norm / f"{sample_id_norm}__{marker_upper}"
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n🧩 Sample: {sample_id_norm}")
    print(f"📄 GeoJSON: {manual_geojson_path}")

    # Bild & Kanäle
    image_path = find_image_for_sample(sample_id_norm)
    print(f"🖼️ Bild: {image_path.name}")
    dapi, mker = load_channels(image_path, chan_dapi=channel_map["DAPI"], chan_marker=channel_map[marker])
    H, W = dapi.shape

    # µm/px lesen → Pixelradien für 7 µm Durchmesser
    mpp_x, mpp_y = get_mpp_from_ome(image_path, fallback=0.249)
    r_um = CIRCLE_DIAMETER_UM / 2.0
    rx_px = r_um / mpp_x
    ry_px = r_um / mpp_y
    print(f"↔ Pixel size: {mpp_x:.4f} µm/px (X), {mpp_y:.4f} µm/px (Y) → Radius ≈ {rx_px:.1f}px × {ry_px:.1f}px")

    # GeoJSON lesen
    with open(manual_geojson_path, "r") as f:
        gj = geojson.load(f)
    features_in = gj["features"] if isinstance(gj, dict) else gj.features

    # Auswertung & gelabelte Maske
    out_props = []
    out_geo_features = []
    labeled_mask = np.zeros((H, W), dtype=np.uint32)

    next_id = 1
    for feat in features_in:
        gobj = shape(feat["geometry"])

        # MultiPoint -> mehrere Ellipsen
        if isinstance(gobj, MultiPoint):
            for pt in gobj.geoms:
                m = geometry_to_mask(pt, H, W, rx_px=rx_px, ry_px=ry_px)
                props = summarize_feature_mask(m, dapi, mker, cell_id=next_id)
                out_props.append(props)
                # Für raw.geojson: Ellipsen-Polygon schreiben (für bestmögliche Kompatibilität)
                ell_poly = point_to_ellipse_polygon(pt.x, pt.y, rx_px, ry_px, resolution=16)
                out_geo_features.append(geojson.Feature(
                    geometry=mapping(ell_poly),
                    properties={"name": marker, "classification": f"{marker}_raw", "cell_id": next_id}
                ))
                labeled_mask[m] = next_id
                next_id += 1
            continue

        # Einzelner Punkt
        if isinstance(gobj, Point):
            m = geometry_to_mask(gobj, H, W, rx_px=rx_px, ry_px=ry_px)
            props = summarize_feature_mask(m, dapi, mker, cell_id=next_id)
            out_props.append(props)
            ell_poly = point_to_ellipse_polygon(gobj.x, gobj.y, rx_px, ry_px, resolution=16)
            out_geo_features.append(geojson.Feature(
                geometry=mapping(ell_poly),
                properties={"name": marker, "classification": f"{marker}_raw", "cell_id": next_id}
            ))
            labeled_mask[m] = next_id
            next_id += 1
            continue

        # Polygon / MultiPolygon
        if isinstance(gobj, (Polygon, MultiPolygon)):
            m = geometry_to_mask(gobj, H, W, rx_px=rx_px, ry_px=ry_px)  # rx/ry ungenutzt hier
            props = summarize_feature_mask(m, dapi, mker, cell_id=next_id)
            out_props.append(props)
            out_geo_features.append(geojson.Feature(
                geometry=mapping(gobj),
                properties={"name": marker, "classification": f"{marker}_raw", "cell_id": next_id}
            ))
            labeled_mask[m] = next_id
            next_id += 1
            continue

        # andere Typen ignorieren

    # === DataFrame robust erzeugen ===
    cols = ["cell_id", "area", f"mean_intensity_{marker.lower()}", "mean_intensity_dapi"]
    if len(out_props) == 0:
        df = pd.DataFrame({c: [] for c in cols})
        print(f"⚠️  {sample_id_norm}: Keine gültigen Features -> leeres DF.")
    else:
        df = pd.DataFrame(out_props)
        for c in cols:
            if c not in df.columns:
                df[c] = []

    # === Speichern: Excel / GeoJSON / Masken (exakte Namen wie Cellpose) ===
    excel_path = save_dir / f"{sample_id_norm}__{marker_upper}_intensity_data.xlsx"
    with pd.ExcelWriter(excel_path, mode="w", engine="openpyxl") as w:
        df.to_excel(w, sheet_name=marker, index=False)
    print(f"✅ Excel: {excel_path.name}")

    raw_geojson_path = save_dir / f"{sample_id_norm}__{marker_upper}_raw.geojson"
    with open(raw_geojson_path, "w") as f:
        geojson.dump(geojson.FeatureCollection(out_geo_features), f)
    print(f"✅ Raw-GeoJSON: {raw_geojson_path.name}")

    # Masken
    mask_png_path = save_dir / f"{sample_id_norm}__{marker_upper}_mask_raw.png"
    Image.fromarray(((labeled_mask > 0).astype(np.uint8) * 255)).save(mask_png_path)
    np.save(save_dir / f"{sample_id_norm}__{marker_upper}_masks.npy", labeled_mask)
    print(f"✅ Masken: {mask_png_path.name}, {sample_id_norm}__{marker_upper}_masks.npy")

    # === Plots (wie im Cellpose-Script) ===
    # Histogram
    plt.figure(figsize=(8, 5))
    if not df.empty:
        sns.histplot(df[f"mean_intensity_{marker.lower()}"], bins=HIST_BINS, binrange=HIST_RANGE,
                     color="darkorange", edgecolor="black")
    else:
        plt.hist([], bins=HIST_BINS, range=HIST_RANGE, edgecolor="black")
    plt.title(f"{marker} Intensity Histogram")
    plt.tight_layout()
    plt.savefig(save_dir / f"{sample_id_norm}__{marker_upper}_histogram_intensity.png", dpi=300)
    plt.close()

    # Scatter DAPI vs Marker
    plt.figure(figsize=(6, 6))
    if not df.empty:
        sns.scatterplot(data=df, x="mean_intensity_dapi", y=f"mean_intensity_{marker.lower()}",
                        edgecolor=None, alpha=0.5)
    else:
        plt.scatter([], [])
    plt.xlabel("Mean DAPI Intensity")
    plt.ylabel(f"Mean {marker} Intensity")
    plt.title(f"{marker} vs DAPI")
    plt.tight_layout()
    plt.savefig(save_dir / f"{sample_id_norm}__{marker_upper}_scatter_dapi_vs_intensity.png", dpi=300)
    plt.close()

def main():
    # Finde alle manuellen GeoJSONs für den Marker
    manual_files = sorted(base_manual_dir.glob(rf"*/*__{marker}/*__{marker}_manual.geojson"))
    if not manual_files:
        raise FileNotFoundError(f"Keine *_manual.geojson unter {base_manual_dir} gefunden.")

    print(f"🔎 Gefundene manuelle GeoJSONs: {len(manual_files)}")
    for p in manual_files:
        try:
            process_manual_geojson(p)
        except Exception as e:
            print(f"❌ Fehler bei {p}: {e}")
        finally:
            gc.collect()

if __name__ == "__main__":
    main()
    print("\n🎉 Fertig. Alle verfügbaren manuellen Cxcl13-Annotationen wurden in identische Outputs überführt.")



🔎 Gefundene manuelle GeoJSONs: 34

🧩 Sample: C10_17
📄 GeoJSON: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cxcl13_images_analyzed\C10_17\C10_17__Cxcl13\C10_17__Cxcl13_manual.geojson
🖼️ Bild: C10_17_CD45-647_Cxcl13-568_Vim-488.ome.tif
↔ Pixel size: 0.2491 µm/px (X), 0.2491 µm/px (Y) → Radius ≈ 14.1px × 14.1px
✅ Excel: C10_17__CXCL13_intensity_data.xlsx
✅ Raw-GeoJSON: C10_17__CXCL13_raw.geojson
✅ Masken: C10_17__CXCL13_mask_raw.png, C10_17__CXCL13_masks.npy

🧩 Sample: C10_32
📄 GeoJSON: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cxcl13_images_analyzed\C10_32\C10_32__Cxcl13\C10_32__Cxcl13_manual.geojson
🖼️ Bild: C10_32_CD45-647_Cxcl13-568_Vim-488.ome.tif
↔ Pixel size: 0.2490 µm/px (X), 0.2490 µm/px (Y) → Radius ≈ 14.1px × 14.1px
✅ Excel: C10_32__CXCL13_intensity_data.xlsx
✅ Raw-GeoJSON: C10_32__CXCL13_raw.geojson
✅ Masken: C10_32__CXCL13_mask_raw.

In [3]:
##Code zum filtrieren - stelle die marker um

# %% 📦 Imports
import gc, time, geojson
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import imageio.v2 as imageio
from skimage.measure import label
import matplotlib.pyplot as plt
import seaborn as sns

Image.MAX_IMAGE_PIXELS = None

# ---------- kleine Log-Hilfe ----------
def log(msg: str):
    from time import strftime
    print(f"[{strftime('%H:%M:%S')}] {msg}")

#Einstellungen
# %% 📍 Marker wählen
marker = "Cxcl13"  # z.B. "Vim", "DAPI", "CD45", "BrdU", "Cd52", "Cxcl13"

# %% 🔍 Batch-Filter: Schwellen manuell setzen
threshold       = 1.0    # z.B. 10.0 (CD45), 10.0 (Vim), 1.0 (Cxcl13)
area_min        = 1     #50 for everything by cxcl13 - cxcl13 was manually annotated; use 1 for manaul
area_max        = 3500   #3500 for everything by cxcl13 - cxcl13 was manually annotated; use 3500 for manaul
dapi_threshold  = 1     #20 for everything by cxcl13 - cxcl13 was manually annotated; use 1 for manaul

# %% 📂 Input/Output
#input_dir ist nur um die Bildnamen zu finden. nichts anderes!!!
input_dir = Path(r"D:\Image_analysis\QuPath_project_overall\export_sections\CD45-647_Cxcl13-568_Vim-488")
output_dir = Path(r"C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cxcl13_images_analyzed")

# 📋 Alle Bilder im Ordner laden
filenames = sorted(input_dir.glob("*.ome.tif"))

# %% 🔢 (nur falls du später mal Kanäle brauchst; hier nicht strikt nötig)
channel_map = {
    "DAPI": 0, "BrdU": 1, "CD45": 1, "Cd52": 2, "Cxcl13": 2, "Vim": 3
}

# %% 🧹 Filter-Funktion
def filter_and_save(output_dir: Path, input_dir: Path, image_stem: str, marker: str,
                    intensity_threshold: float | None = None,
                    area_min: float | None = None,
                    area_max: float | None = None,
                    dapi_threshold: float | None = None):

    marker_upper = marker.upper()
    marker_lower = marker.lower()
    save_dir = output_dir / image_stem / f"{image_stem}__{marker_upper}"
    log(f"📎 Filtering {image_stem}/{marker_upper} @ I≥{intensity_threshold}, area∈[{area_min},{area_max}], DAPI≥{dapi_threshold}")

    if not save_dir.exists():
        raise FileNotFoundError(f"❌ Ordner fehlt: {save_dir}")

    # ---- Excel laden
    excel_path = save_dir / f"{image_stem}__{marker_upper}_intensity_data.xlsx"
    if not excel_path.exists():
        raise FileNotFoundError(f"❌ Excel fehlt: {excel_path}")
    df = pd.read_excel(excel_path, sheet_name=marker)
    log(f"   • Excel geladen: n={len(df)} Zeilen")

    # ---- Filter berechnen
    df["manual_filter_intensity"] = True if intensity_threshold is None else (df[f"mean_intensity_{marker_lower}"] >= intensity_threshold)
    df["manual_filter_area"] = True
    if area_min is not None: df["manual_filter_area"] &= df["area"] >= area_min
    if area_max is not None: df["manual_filter_area"] &= df["area"] <= area_max
    df["manual_filter_dapi"] = True if dapi_threshold is None else (df["mean_intensity_dapi"] >= dapi_threshold)

    df["manual_filter_combined"] = df["manual_filter_intensity"] & df["manual_filter_area"] & df["manual_filter_dapi"]
    kept_ids = df.loc[df["manual_filter_combined"], "cell_id"].astype(int).tolist()
    df[f"manual_filter_{marker_lower}"] = df["manual_filter_combined"]

    # temporäre Spalten aufräumen
    drop_cols = [c for c in df.columns if c.startswith("manual_filter_") and c != f"manual_filter_{marker_lower}"]
    df.drop(columns=drop_cols, inplace=True, errors="ignore")

    # zurückschreiben
    with pd.ExcelWriter(excel_path, mode="a", engine="openpyxl", if_sheet_exists="replace") as w:
        df.to_excel(w, sheet_name=marker, index=False)
    log(f"   • Filter gespeichert (Excel Sheet '{marker}'); behalten: {df[f'manual_filter_{marker_lower}'].sum()}/{len(df)}")

    # ---- Masken laden & gefilterte Maske schreiben
    mask_path = save_dir / f"{image_stem}__{marker_upper}_mask_raw.png"
    if not mask_path.exists():
        raise FileNotFoundError(f"❌ mask_raw fehlt: {mask_path}")
    raw_mask = imageio.imread(mask_path) > 0
    labeled_mask = label(raw_mask)
    filtered_mask = np.isin(labeled_mask, np.array(kept_ids, dtype=int)).astype(np.uint8) * 255
    filtered_mask_path = save_dir / f"{image_stem}__{marker_upper}_filtered_mask.png"
    Image.fromarray(filtered_mask).save(filtered_mask_path)
    log(f"   • Gefilterte Maske: {filtered_mask_path.name}")

    # ---- GeoJSON filtern (nur wenn raw vorhanden)
    raw_geojson_path = save_dir / f"{image_stem}__{marker_upper}_raw.geojson"
    if raw_geojson_path.exists():
        with open(raw_geojson_path, "r") as f:
            raw_data = geojson.load(f)

        feats = []
        for feat in raw_data.get("features", []):
            cid = feat.get("properties", {}).get("cell_id")
            if cid in kept_ids:
                feat["properties"]["classification"] = f"{marker}_filtered"
                feats.append(feat)

        filt_geojson_path = save_dir / f"{image_stem}__{marker_upper}_filtered.geojson"
        with open(filt_geojson_path, "w") as f:
            geojson.dump(geojson.FeatureCollection(feats), f)
        log(f"   • GeoJSON gespeichert: {filt_geojson_path.name} | nFeatures={len(feats)}")
    else:
        log(f"   ⚠️ RAW‑GeoJSON fehlt ({raw_geojson_path.name}); überspringe GeoJSON‑Export.")

    # ---- Plots
    # Scatter area vs marker
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=df, x="area", y=f"mean_intensity_{marker_lower}",
        hue=df[f"manual_filter_{marker_lower}"], palette={True: "red", False: "blue"},
        s=60, edgecolor="black", alpha=0.6
    )
    if area_min is not None: plt.axvline(area_min, color="gray", linestyle="--", linewidth=1.5, label=f"area_min={area_min}")
    if area_max is not None: plt.axvline(area_max, color="gray", linestyle="--", linewidth=1.5, label=f"area_max={area_max}")
    if intensity_threshold is not None: plt.axhline(intensity_threshold, color="gray", linestyle="--", linewidth=1.5, label=f"{marker}≥{intensity_threshold}")
    plt.xlabel("Zellfläche (px²)"); plt.ylabel(f"Mittlere Intensität: {marker}")
    plt.title(f"{marker}: Fläche vs. Intensität (Filter)"); plt.legend(title="Behalten?"); plt.grid(True); plt.tight_layout()
    scatter_path = save_dir / f"{image_stem}__{marker_upper}_scatter_area_vs_intensity.png"
    plt.savefig(scatter_path, dpi=300); plt.close()
    log(f"   • Scatterplot: {scatter_path.name}")

    # Marker‑Histogramm
    plt.figure(figsize=(8, 5))
    sns.histplot(df[f"mean_intensity_{marker_lower}"], bins=255, binrange=(0, 255), color="darkorange", edgecolor="black")
    if intensity_threshold is not None:
        plt.axvline(intensity_threshold, color="red", linestyle="--", linewidth=2, label=f"Thr={intensity_threshold}")
        plt.legend()
    plt.title(f"{marker} Intensity Histogram (mit Schwelle)")
    plt.xlabel(f"Mittlere Intensität: {marker}"); plt.ylabel("Zellanzahl"); plt.tight_layout()
    hist_path = save_dir / f"{image_stem}__{marker_upper}_histogram_intensity.png"
    plt.savefig(hist_path, dpi=300); plt.close()
    log(f"   • Marker‑Histogramm: {hist_path.name}")

    # DAPI‑Histogramm
    plt.figure(figsize=(8, 5))
    sns.histplot(df["mean_intensity_dapi"], bins=255, binrange=(0, 255), color="slateblue", edgecolor="black")
    if dapi_threshold is not None:
        plt.axvline(dapi_threshold, color="gray", linestyle="--", linewidth=2, label=f"DAPI≥{dapi_threshold}")
        plt.legend()
    plt.title("DAPI Intensity Histogram (mit Schwelle)")
    plt.xlabel("Mittlere Intensität: DAPI"); plt.ylabel("Zellanzahl"); plt.tight_layout()
    hist_dapi_path = save_dir / f"{image_stem}__{marker_upper}_histogram_dapi_intensity.png"
    plt.savefig(hist_dapi_path, dpi=300); plt.close()
    log(f"   • DAPI‑Histogramm: {hist_dapi_path.name}")

    log(f"✅ {image_stem}/{marker_upper}: behalten {len(kept_ids)} / {len(df)}")



for path in filenames:
    full_stem  = path.stem.replace(".ome", "")            # z. B. "C6_33_CD45-647_Cxcl13-568_Vim-488"
    image_stem = "_".join(full_stem.split("_")[0:2])      # → "C6_33"
    try:
        filter_and_save(
            output_dir=output_dir,
            input_dir=input_dir,
            image_stem=image_stem,     # ⚠️ Funktionsargument heißt image_stem
            marker=marker,
            intensity_threshold=threshold,
            area_min=area_min,
            area_max=area_max,
            dapi_threshold=dapi_threshold
        )
    except Exception as e:
        log(f"❌ Fehler bei {image_stem}: {e}")
    finally:
        gc.collect()

log("🎉 Hey, endlich ganz fertig!")


[15:39:53] 📎 Filtering C10_17/CXCL13 @ I≥1.0, area∈[1,3500], DAPI≥1
[15:39:53]    • Excel geladen: n=15 Zeilen
[15:39:53]    • Filter gespeichert (Excel Sheet 'Cxcl13'); behalten: 15/15
[15:39:58]    • Gefilterte Maske: C10_17__CXCL13_filtered_mask.png
[15:39:58]    • GeoJSON gespeichert: C10_17__CXCL13_filtered.geojson | nFeatures=15
[15:39:58]    • Scatterplot: C10_17__CXCL13_scatter_area_vs_intensity.png
[15:39:58]    • Marker‑Histogramm: C10_17__CXCL13_histogram_intensity.png
[15:39:59]    • DAPI‑Histogramm: C10_17__CXCL13_histogram_dapi_intensity.png
[15:39:59] ✅ C10_17/CXCL13: behalten 15 / 15
[15:39:59] 📎 Filtering C10_32/CXCL13 @ I≥1.0, area∈[1,3500], DAPI≥1
[15:39:59]    • Excel geladen: n=20 Zeilen
[15:39:59]    • Filter gespeichert (Excel Sheet 'Cxcl13'); behalten: 20/20
[15:40:05]    • Gefilterte Maske: C10_32__CXCL13_filtered_mask.png
[15:40:05]    • GeoJSON gespeichert: C10_32__CXCL13_filtered.geojson | nFeatures=20
[15:40:05]    • Scatterplot: C10_32__CXCL13_scatter_area

In [ ]:
#analyse code!! diese code kombiniert die geojson berechnet ueberlap fuer meine Marker

from pathlib import Path
from collections import Counter
import json
import math
import numpy as np  # wichtig für STRtree-Index-Rückgaben (np.int64)
import pandas as pd

from shapely.geometry import shape, mapping
from shapely.strtree import STRtree

# ---- Für Unschärfe-Score (DAPI aus OME-TIF laden) ----
from tifffile import imread as tif_imread
from skimage.filters import laplace

# -------------------- KONFIGURATION --------------------
# Oberordner, der die Bild-Ordner (C6_19, I7_18, …) enthält
ROOT = Path(r"C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cd52_images_analyzed2")
# Zielordner für kombinierte Outputs & Excel
OUTPUT_ROOT = Path(r"C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cd52_images_analyzed2")

# Marker-Namen (logisch) und erlaubte Ordner-Aliase (Suffix nach "<stem>__")
MARKER_ALIASES = {
    "DAPI": ["DAPI"],
    "CD45": ["CD45", "Cd45"], # case-tolerant
    "Vim":  ["VIM", "Vim"],   # case-tolerant
    "CD52": ["CD52", "Cd52"], # case-tolerant
}

# Schwellen (τ) für Coverage und DAPI-Puffer (Pixel) pro Marker
TAU = {
    "CD45": 0.90,
    "Vim":  0.92,
    "CD52": 0.90,
}
DAPI_BUFFER_PX = {
    "CD45": 2,
    "Vim":  2,
    "CD52": 2,
}

# Rettungsanker: DAPI-Centroid ∈ Marker → positiv (auch wenn Coverage knapp unter τ)
USE_CENTROID_BACKUP = True

# Sicherheitsgurt: max. Centroid-Distanz ≤ faktor * DAPI-Äquivalentradius (marker-spezifisch)
DIST_FACTOR = {
    "CD45": 1.5,
    "Vim":  1.5,
    "CD52": 1.5,
}

# Mini-Gate: wenn echte Lokalität erkennbar ist, Distanzgurt nicht blockierend anwenden
# (Verhindert fälschliche Ablehnung bei großen zytoplasmatischen Masken)
TAU_GATE_MIN = 0.05  # 5% Mini-Overlap oder Centroid-in genügt, um Distanzgurt zu „übergehen“

# Idempotenz: Bild überspringen, wenn bereits combined.geojson existiert
SKIP_IF_COMBINED_EXISTS = True

# Wie der Anzeigename aussehen soll:
# - "positives_only": nur die Marker mit "+" (z. B. "CD45+ Vim+")
# - "full_combo":     vollständige Kombi (z. B. "DAPI+ CD45+ Vim- CD52+")
NAME_STYLE = "positives_only"

# Marker-Polygone zusätzlich sichtbar mitschreiben?
INCLUDE_MARKER_GEOMETRIES = False  # bei Bedarf True setzen

# -------------------- BILDSUCHE / DAPI-KANAL (für Unschärfe) --------------------
# Gib hier 1..n Basisordner an, in denen die OME-TIFs liegen könnten.
# Wir suchen rekursiv nach Dateien, deren Name den STEM enthält und auf .ome.tif endet.
IMAGE_SEARCH_DIRS = [
    Path(r"D:\Image_analysis\QuPath_project_overall\export_sections"),
    # weitere Ordner nach Bedarf anhängen ...
]
# Wenn deine OME-TIFs konsistent sind, kannst du DAPI hier fix verdrahten.
# AUTO_ERKENNUNG versucht erst (C,Y,X) vs (Y,X,C) zu raten und nimmt Kanal 0 als DAPI.
# Optional: DAPI_CHANNEL_INDEX explizit setzen (z. B. 0).
AUTO_DETECT_CHANNEL_ORDER = True
DAPI_CHANNEL_INDEX = 0  # wird genutzt, wenn AUTO_DETECT_CHANNEL_ORDER True ist (wir nehmen dann Kanal 0)

# -------------------- QC-PARAMETER --------------------
# 1%-Grenze: maximal 1% DAPI-Zellen dürfen dieselbe Marker-Maske mit einer anderen DAPI-Zelle teilen
MULTI_ASSIGN_LIMIT = 0.01  # 1%


# -------------------- HILFSFUNKTIONEN --------------------
def read_geojson(path: Path):
    """GeoJSON laden → Listen von Geometrien und Properties."""
    data = json.loads(path.read_text(encoding="utf-8"))
    feats = data.get("features", [])
    geoms, props = [], []
    for f in feats:
        geoms.append(shape(f["geometry"]))
        props.append(f.get("properties", {}) or {})
    return geoms, props

def write_geojson(path: Path, features):
    """GeoJSON schreiben."""
    out = {"type": "FeatureCollection", "features": features}
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(out), encoding="utf-8")

def dapi_equiv_radius(area: float) -> float:
    """Äquivalentradius r für Kreis mit gleicher Fläche wie DAPI."""
    return math.sqrt(area / math.pi) if area > 0 else 0.0

def centroid_distance_ok(poly_dapi, poly_marker, factor: float):
    """
    Sicherheitsgurt prüfen:
    - Liefert (ok: bool, dist: float, r_equiv: float)
    - ok == True, wenn Distanz ≤ factor * r_equiv
    """
    r = dapi_equiv_radius(poly_dapi.area)
    if r <= 0:
        return True, 0.0, 0.0
    d = poly_dapi.centroid.distance(poly_marker.centroid)
    return (d <= factor * r), d, r

def coverage_fraction(dapi_poly, marker_poly, buffer_px=0):
    """Coverage = Fläche(DAPI_buffer ∩ Marker) / Fläche(DAPI_buffer)."""
    dapi_b = dapi_poly.buffer(buffer_px) if buffer_px and buffer_px > 0 else dapi_poly
    if not dapi_b.intersects(marker_poly):
        return 0.0
    inter = dapi_b.intersection(marker_poly)
    if inter.is_empty:
        return 0.0
    A = dapi_b.area
    return inter.area / A if A > 0 else 0.0

def build_index(geoms):
    """STRtree-Index + Rückmap von Geom-ID → Listenindex."""
    tree = STRtree(geoms)
    idmap = {id(g): i for i, g in enumerate(geoms)}
    return tree, idmap

def find_marker_folder(stem_dir: Path, stem: str, aliases):
    """Suche Unterordner <stem>__ALIAS (robust gegenüber Groß-/Kleinschreibung)."""
    for alias in aliases:
        cand = stem_dir / f"{stem}__{alias}"
        if cand.exists():
            return cand
    # Case-insensitive Fallback
    lower_aliases = [a.lower() for a in aliases]
    for sub in stem_dir.iterdir():
        if sub.is_dir() and sub.name.lower().startswith(f"{stem.lower()}__"):
            suffix = sub.name.split("__", 1)[1].lower()
            if suffix in lower_aliases:
                return sub
    return None

def find_filtered_geojson(marker_dir: Path, stem: str, alias: str):
    """Erwarte <stem>__<alias>_filtered.geojson im Marker-Ordner (mehrere Schreibweisen)."""
    candidates = [
        marker_dir / f"{stem}__{alias}_filtered.geojson",
        marker_dir / f"{stem}__{alias.upper()}_filtered.geojson",
        marker_dir / f"{stem}__{alias.capitalize()}_filtered.geojson",
    ]
    for p in candidates:
        if p.exists():
            return p
    # letzte Option: irgendein *_filtered.geojson im Ordner
    found = list(marker_dir.glob("*_filtered.geojson"))
    return found[0] if found else None

def combo_label(flags: dict):
    """Kombinationslabel aus Marker-Flags (DAPI ist per Definition positiv)."""
    parts = ["DAPI+"]
    parts.append("CD45+" if flags.get("CD45") else "CD45-")
    parts.append("Vim+"  if flags.get("Vim")  else "Vim-")
    parts.append("CD52+" if flags.get("CD52") else "CD52-")
    return " ".join(parts)

def build_display_name(flags: dict, style: str = "positives_only"):
    """
    Erzeuge Anzeigename für die DAPI-Zelle.
    style: "positives_only" = nur positive Marker (z. B. "CD45+ Vim+")
           "full_combo"     = vollständige Kombi (z. B. "DAPI+ CD45- Vim+ CD52-")
    """
    if style == "full_combo":
        return combo_label(flags)
    # positives_only
    pos = []
    for m in ["CD45", "Vim", "CD52"]:
        if flags.get(m):
            pos.append(f"{m}+")
    return " ".join(pos) if pos else ""  # leer bei keiner Positivität


# -------------------- BILD-LADEN & UNSCHÄRFE (Laplacian-Varianz) --------------------
def find_image_file(stem: str):
    """Suche rekursiv nach einer .ome.tif, deren Dateiname den STEM enthält."""
    stem_lower = stem.lower()
    for root in IMAGE_SEARCH_DIRS:
        if not root.exists():
            continue
        # Suche breit: **/*<stem>*.ome.tif
        for p in root.rglob("*.ome.tif"):
            if stem_lower in p.name.lower():
                return p
    return None

def load_dapi_plane(image_path: Path):
    """
    Lädt den DAPI-Kanal als 2D-Array (float32 0..1).
    Heuristik: AUTO_DETECT_CHANNEL_ORDER → (C,Y,X) wenn Achse0 klein (<=6), sonst (Y,X,C).
    DAPI_CHANNEL_INDEX wird verwendet (Default 0).
    """
    arr = tif_imread(str(image_path))

    # Reduziere ggf. Z/T auf Einzelbild
    # Übliche OME-Layouts: (Y,X,C), (C,Y,X), (Z,Y,X,C), (T,Y,X,C), (T,Z,C,Y,X), ...
    # Wir ziehen die letzten 3 Achsen als (Y,X,C) oder (C,Y,X) heran.
    a = np.squeeze(arr)
    if a.ndim == 2:
        yx = a.astype(np.float32)
        return normalize01(yx)

    if a.ndim == 3:
        s0, s1, s2 = a.shape
        if AUTO_DETECT_CHANNEL_ORDER and s0 <= 6 and s0 < s1 and s0 < s2:
            # (C, Y, X)
            dapi = a[DAPI_CHANNEL_INDEX]
        else:
            # (Y, X, C)
            dapi = a[..., DAPI_CHANNEL_INDEX]
        return normalize01(dapi.astype(np.float32))

    # 4D oder 5D: versuche, auf (C,Y,X) oder (Y,X,C) zu projizieren
    # Wir nehmen die erste(n) Scheiben für T/Z.
    while a.ndim > 3:
        a = a[0]
    # nun 3D:
    s0, s1, s2 = a.shape
    if AUTO_DETECT_CHANNEL_ORDER and s0 <= 6 and s0 < s1 and s0 < s2:
        dapi = a[DAPI_CHANNEL_INDEX]
    else:
        dapi = a[..., DAPI_CHANNEL_INDEX]
    return normalize01(dapi.astype(np.float32))

def normalize01(img: np.ndarray):
    vmin = float(np.nanpercentile(img, 0.5))
    vmax = float(np.nanpercentile(img, 99.5))
    if vmax <= vmin:
        vmax = vmin + 1.0
    img = np.clip((img - vmin) / (vmax - vmin), 0.0, 1.0)
    return img.astype(np.float32)

def laplacian_variance_dapi(image_path: Path):
    """
    Berechnet die Laplacian-Varianz (Fokus/Schärfe) des DAPI-Kanals.
    Höher = schärfer. Gibt float oder np.nan bei Fehler zurück.
    """
    try:
        dapi = load_dapi_plane(image_path)
        lap = laplace(dapi)  # skimage Laplace
        var = float(np.var(lap))
        return var
    except Exception as e:
        print(f"⚠️ Laplacian-Varianz konnte nicht berechnet werden für {image_path}: {e}")
        return np.nan


# -------------------- QC-HELPER --------------------
def robust_z(x_vals):
    x = np.asarray(x_vals, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) or 1e-9
    return (x - med) / (1.4826 * mad), med, mad

def pct(numer, denom):
    return 0.0 if denom == 0 else 100.0 * float(numer) / float(denom)

def multi_assign_percent(marker_match_list, total_cells):
    # marker_match_list: Liste der Werte aus props[f"{m}_match"] (kann None enthalten)
    ids = [m for m in marker_match_list if m is not None]
    if total_cells == 0 or not ids:
        return 0.0
    c = Counter(ids)
    # Zellen, deren match-ID von >=2 Zellen benutzt wurde
    involved = sum(cnt for _, cnt in c.items() if cnt >= 2)
    return pct(involved, total_cells)

def fragile_positive_percent(rows, marker, tau):
    # rows: Iterable über Zell-Properties-Dicts
    pos = [r for r in rows if r.get(f"{marker}_pos", False)]
    if not pos:
        return 0.0
    fragile = [r for r in pos if (r.get(f"{marker}_coverage", 0.0) < tau and r.get(f"{marker}_centroid", False))]
    return pct(len(fragile), len(pos))

def dist_ratio_median(rows, marker):
    vals = []
    for r in rows:
        if r.get(f"{marker}_pos", False):
            d = r.get(f"{marker}_centroid_dist", None)
            req = r.get(f"{marker}_dapi_req_radius", None)
            if d is not None and req not in (None, 0.0):
                vals.append(d / req)
    return float(np.nanmedian(vals)) if vals else np.nan

def coverage_median(rows, marker):
    vals = [r.get(f"{marker}_coverage", np.nan) for r in rows if r.get(f"{marker}_pos", False)]
    return float(np.nanmedian(vals)) if vals else np.nan

def blur_proxy_score(rows, marker, tau):
    # 1) hoher Anteil Fragile-Positiva  2) niedriger Coverage-Median  3) hoher Distanz/Radius
    fp = fragile_positive_percent(rows, marker, tau) / 100.0          # 0..1
    cov_med = coverage_median(rows, marker)                            # 0..1
    cov_term = max(0.0, (tau - (cov_med if not np.isnan(cov_med) else tau)))
    dr = dist_ratio_median(rows, marker)                               # ~1 gut, >1.5 schlecht
    dr_term = 0.0 if np.isnan(dr) else max(0.0, (dr - 1.0)) / 0.5
    return float(0.5*fp + 0.3*cov_term + 0.2*dr_term)

def positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, tau, buffer_px, marker_name):
    geoms = marker_geoms.get(marker_name, [])
    tree, idmap = marker_idx.get(marker_name, (None, None))
    if not geoms or tree is None:
        return 0.0
    total = len(dapi_geoms)
    if total == 0:
        return 0.0
    pos = 0
    for poly_dapi in dapi_geoms:
        query_geom = poly_dapi.buffer(buffer_px) if buffer_px else poly_dapi
        best_cov = 0.0
        candidates = tree.query(query_geom)
        hit = False
        for cand in candidates:
            if isinstance(cand, (int, np.integer)):
                idx = int(cand); cand_geom = geoms[idx]
            else:
                cand_geom = cand
            cov = coverage_fraction(poly_dapi, cand_geom, buffer_px=buffer_px)
            centroid_hit = poly_dapi.centroid.within(cand_geom)
            if not centroid_hit and cov < TAU_GATE_MIN:
                ok_dist, _, _ = centroid_distance_ok(poly_dapi, cand_geom, DIST_FACTOR[marker_name])
                if not ok_dist:
                    continue
            if (cov >= tau) or (USE_CENTROID_BACKUP and centroid_hit):
                if cov > best_cov:
                    best_cov = cov
                    hit = True
        if hit:
            pos += 1
    return pct(pos, total)


# -------------------- KERNLOGIK --------------------
def classify_stem(stem_dir: Path):
    """
    Klassifiziert ein Bild (Ordner <stem_dir>):
    - Lädt DAPI + Marker (gefilterte GeoJSONs)
    - Wendet Coverage-τ + Centroid-Backup + Sicherheitsgurt an
    - Schreibt combined.geojson & combocounts.csv
    - Berechnet QC-Metriken & What-If & DAPI-Laplacian-Varianz
    - Gibt alle Strukturen zurück (für Aggregate & Big-Excel)
    """
    stem = stem_dir.name
    out_dir = OUTPUT_ROOT / stem
    combined_path = out_dir / f"{stem}__combined.geojson"
    counts_path   = out_dir / f"{stem}__combocounts.csv"
    per_image_xlsx = out_dir / f"{stem}__per_image_QC.xlsx"

    if SKIP_IF_COMBINED_EXISTS and combined_path.exists():
        print(f"♻️  Überschreibe bestehende Outputs für: {stem}")


    # --- DAPI laden ---
    dapi_dir = find_marker_folder(stem_dir, stem, MARKER_ALIASES["DAPI"])
    if dapi_dir is None:
        raise FileNotFoundError(f"DAPI-Ordner fehlt für {stem} unter {stem_dir}")
    dapi_gj = find_filtered_geojson(dapi_dir, stem, "DAPI")
    if dapi_gj is None or not dapi_gj.exists():
        raise FileNotFoundError(f"DAPI *_filtered.geojson fehlt in {dapi_dir}")
    dapi_geoms, dapi_props = read_geojson(dapi_gj)

    # --- Marker laden + indizieren ---
    marker_geoms = {}
    marker_props = {}
    marker_idx   = {}

    for logical_name, aliases in MARKER_ALIASES.items():
        if logical_name == "DAPI":
            continue
        mdir = find_marker_folder(stem_dir, stem, aliases)
        if mdir is None:
            print(f"⚠️  {logical_name}: Ordner fehlt für {stem}")
            marker_geoms[logical_name] = []
            marker_props[logical_name] = []
            marker_idx[logical_name] = (None, None)
            continue
        alias_hint = aliases[0]
        mgj = find_filtered_geojson(mdir, stem, alias_hint)
        if mgj is None or not mgj.exists():
            print(f"⚠️  {logical_name}: *_filtered.geojson fehlt in {mdir}")
            marker_geoms[logical_name] = []
            marker_props[logical_name] = []
            marker_idx[logical_name] = (None, None)
            continue
        g, p = read_geojson(mgj)
        marker_geoms[logical_name] = g
        marker_props[logical_name] = p
        marker_idx[logical_name]   = build_index(g) if g else (None, None)

    # Kurzer Diagnose-Print
    print(f"[{stem}] DAPI: {len(dapi_geoms)}  "
          f"CD45: {len(marker_geoms.get('CD45', []))}  "
          f"Vim: {len(marker_geoms.get('Vim', []))}  "
          f"CD52: {len(marker_geoms.get('CD52', []))}")

    # --- Klassifikation pro DAPI-Zelle ---
    out_features = []
    combos = Counter()
    pos_cd45 = pos_vim = pos_cd52 = 0
    total_cells = len(dapi_geoms)

    # Für QC: Listen der match-IDs je Marker
    match_ids = {"CD45": [], "Vim": [], "CD52": []}

    for i, poly_dapi in enumerate(dapi_geoms):
        props = dict(dapi_props[i])
        props["DAPI"] = True

        flags = {}
        metrics = {}

        for m in ["CD45", "Vim", "CD52"]:
            geoms = marker_geoms.get(m, [])
            tree, idmap = marker_idx.get(m, (None, None))
            tau = TAU[m]
            buf = DAPI_BUFFER_PX[m]
            dist_factor = DIST_FACTOR[m]

            best_cov = 0.0
            best_idx = None
            best_centroid_hit = False
            best_dist = None
            best_req_radius = None

            if geoms and tree is not None:
                query_geom = poly_dapi.buffer(buf) if buf else poly_dapi
                candidates = tree.query(query_geom)

                for cand in candidates:
                    if isinstance(cand, (int, np.integer)):
                        idx = int(cand)
                        cand_geom = geoms[idx]
                    else:
                        cand_geom = cand
                        idx = idmap[id(cand_geom)] if idmap is not None else geoms.index(cand_geom)

                    cov = coverage_fraction(poly_dapi, cand_geom, buffer_px=buf)
                    centroid_hit = poly_dapi.centroid.within(cand_geom)

                    if not centroid_hit and cov < TAU_GATE_MIN:
                        ok_dist, dcent, req_r = centroid_distance_ok(poly_dapi, cand_geom, dist_factor)
                        if not ok_dist:
                            continue
                    else:
                        ok_dist, dcent, req_r = centroid_distance_ok(poly_dapi, cand_geom, dist_factor)

                    if (cov >= tau) or (USE_CENTROID_BACKUP and centroid_hit):
                        if cov > best_cov:
                            best_cov = cov
                            best_idx = idx
                            best_centroid_hit = centroid_hit
                            best_dist = dcent
                            best_req_radius = req_r

            is_pos = best_idx is not None
            flags[m] = is_pos

            metrics[f"{m}_coverage"] = round(best_cov, 4)
            metrics[f"{m}_centroid"] = bool(best_centroid_hit)
            metrics[f"{m}_centroid_dist"] = None if best_dist is None else round(best_dist, 3)
            metrics[f"{m}_dapi_req_radius"] = None if best_req_radius is None else round(best_req_radius, 3)

            match_val = None
            if is_pos:
                base_props = marker_props[m][best_idx] if (m in marker_props and best_idx is not None and best_idx < len(marker_props[m])) else {}
                match_val = base_props.get("id") or base_props.get("label") or best_idx
            metrics[f"{m}_match"] = match_val
            match_ids[m].append(match_val)

        props.update(metrics)
        for m in ["CD45", "Vim", "CD52"]:
            props[f"{m}_pos"] = bool(flags[m])

        combo = combo_label(flags)
        props["combo"] = combo
        props["name"]  = build_display_name(flags, NAME_STYLE)
        props["label"] = props["name"]

        out_features.append({
            "type": "Feature",
            "geometry": mapping(poly_dapi),
            "properties": props
        })
        combos[combo] += 1
        if flags.get("CD45"): pos_cd45 += 1
        if flags.get("Vim"):  pos_vim  += 1
        if flags.get("CD52"): pos_cd52 += 1

    # Optional: Marker-Geometrien mitspeichern
    if INCLUDE_MARKER_GEOMETRIES:
        for m in ["CD45", "Vim", "CD52"]:
            geoms = marker_geoms.get(m, [])
            props_list = marker_props.get(m, [])
            for gi, g in enumerate(geoms):
                basep = props_list[gi] if gi < len(props_list) else {}
                extra = {"layer": m, "name": m, "label": m}
                out_features.append({
                    "type": "Feature",
                    "geometry": mapping(g),
                    "properties": {**basep, **extra}
                })

    # --- Outputs pro Bild schreiben (GeoJSON/CSV) ---
    out_dir.mkdir(parents=True, exist_ok=True)
    write_geojson(combined_path, out_features)
    lines = ["combo,count"] + [f"{k},{v}" for k, v in sorted(combos.items())]
    counts_path.write_text("\n".join(lines), encoding="utf-8")

    print(f"✅ Geschrieben: {combined_path}")
    print(f"✅ Geschrieben: {counts_path}")

    marker_counts = {
        "stem": stem,
        "total_cells": total_cells,
        "CD45_pos": pos_cd45,
        "CD45_neg": total_cells - pos_cd45,
        "Vim_pos":  pos_vim,
        "Vim_neg":  total_cells - pos_vim,
        "CD52_pos": pos_cd52,
        "CD52_neg": total_cells - pos_cd52,
    }

    # -------------------- QC-METRIKEN (pro Bild) --------------------
    rows_props = [f["properties"] for f in out_features]  # nur DAPI-Features

    # Unschärfe-Score (Laplacian-Varianz) aus DAPI-Kanal
    img_path = find_image_file(stem)
    if img_path is None:
        print(f"⚠️ Kein OME-TIF zum STEM gefunden (für Unschärfe): {stem}")
        lap_var = np.nan
    else:
        lap_var = laplacian_variance_dapi(img_path)

    qc = {
        "stem": stem,
        "total_cells": total_cells,
        # Fragile-Positiva (%)
        "CD45_fragile_pct": fragile_positive_percent(rows_props, "CD45", TAU["CD45"]),
        "Vim_fragile_pct":  fragile_positive_percent(rows_props, "Vim",  TAU["Vim"]),
        "CD52_fragile_pct": fragile_positive_percent(rows_props, "CD52", TAU["CD52"]),
        # Multi-Assign (% DAPI-Zellen involviert)
        "multi_assign_pct_CD45": multi_assign_percent(match_ids["CD45"], total_cells),
        "multi_assign_pct_Vim":  multi_assign_percent(match_ids["Vim"],  total_cells),
        "multi_assign_pct_CD52": multi_assign_percent(match_ids["CD52"], total_cells),
        # Distanz/Radius (Median)
        "centroid_dist_ratio_median_CD45": dist_ratio_median(rows_props, "CD45"),
        "centroid_dist_ratio_median_Vim":  dist_ratio_median(rows_props, "Vim"),
        "centroid_dist_ratio_median_CD52": dist_ratio_median(rows_props, "CD52"),
        # Coverage-Median (nur Positiva)
        "coverage_median_CD45": coverage_median(rows_props, "CD45"),
        "coverage_median_Vim":  coverage_median(rows_props, "Vim"),
        "coverage_median_CD52": coverage_median(rows_props, "CD52"),
        # Alle Marker positiv
        "marker_all_pos_pct": pct(sum(1 for r in rows_props if r.get("CD45_pos") and r.get("Vim_pos") and r.get("CD52_pos")), total_cells),
        # Blur-Proxies
        "blur_proxy_Vim":  blur_proxy_score(rows_props, "Vim",  TAU["Vim"]),
        "blur_proxy_CD45": blur_proxy_score(rows_props, "CD45", TAU["CD45"]),
        "blur_proxy_CD52": blur_proxy_score(rows_props, "CD52", TAU["CD52"]),
        # Reale Schärfemessung aus Bild
        "laplacian_var_dapi": lap_var,
        "image_path": str(img_path) if img_path else ""
    }
    # Flags (1%-Regel)
    qc["flag_multi_assign"] = any(qc[k] > (MULTI_ASSIGN_LIMIT*100.0) for k in (
        "multi_assign_pct_CD45", "multi_assign_pct_Vim", "multi_assign_pct_CD52"
    ))

    # -------------------- WHAT-IF (pro Bild) --------------------
    whatif_rows = []
    for m in ["CD45", "Vim", "CD52"]:
        base_tau = TAU[m]
        base_buf = DAPI_BUFFER_PX[m]
        recs = {
            "marker": m,
            f"PosRate_buf{base_buf-1 if base_buf>0 else 0}": positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, base_tau, max(0, base_buf-1), m),
            f"PosRate_buf{base_buf}":     positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, base_tau, base_buf, m),
            f"PosRate_buf{base_buf+1}":   positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, base_tau, base_buf+1, m),
            "PosRate_tau_minus0.02":      positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, max(0.0, base_tau-0.02), base_buf, m),
            "PosRate_tau_plus0.02":       positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, min(1.0, base_tau+0.02), base_buf, m),
        }
        whatif_rows.append(recs)
    df_whatif = pd.DataFrame(whatif_rows)

    # ---- per-Image-Excel (nur QC) schreiben ----
    with pd.ExcelWriter(per_image_xlsx) as xw:
        pd.DataFrame([qc]).to_excel(xw, index=False, sheet_name="QC_metrics")
        df_whatif.to_excel(xw, index=False, sheet_name="QC_what_if")
    print(f"🧪 QC-Excel geschrieben: {per_image_xlsx}")

    return {
        "stem": stem,
        "combos": combos,
        "marker_counts": marker_counts,
        "qc": qc,
        "whatif": df_whatif
    }


def main():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    stems = [p for p in ROOT.iterdir() if p.is_dir()]
    if not stems:
        print(f"⚠️ Keine Bild-Ordner unter {ROOT} gefunden")
        return

    print(f"🔎 {len(stems)} Bild-Ordner gefunden. Starte Verarbeitung…")

    rows_combos = []          # (stem, combo, count)
    rows_marker_counts = []   # per stem
    rows_qc = []              # per stem QC
    whatif_all = []           # pro stem (append marker-wise rows + stem)

    for stem_dir in stems:
        try:
            res = classify_stem(stem_dir)
            stem = res["stem"]
            combos = res["combos"]
            marker_counts = res["marker_counts"]
            qc = res["qc"]
            df_whatif = res["whatif"]

            for combo, count in combos.items():
                rows_combos.append({"stem": stem, "combo": combo, "count": count})
            if marker_counts:
                rows_marker_counts.append(marker_counts)
            if qc:
                rows_qc.append(qc)
            if isinstance(df_whatif, pd.DataFrame) and not df_whatif.empty:
                tmp = df_whatif.copy()
                tmp.insert(0, "stem", stem)
                whatif_all.append(tmp)

        except Exception as e:
            print(f"❌ {stem_dir.name}: {e}")

    if not rows_combos and not rows_marker_counts and not rows_qc:
        print("⚠️ Keine Daten zum Schreiben in Excel.")
        return

    final_xlsx = OUTPUT_ROOT / "__final_counts.xlsx"
    with pd.ExcelWriter(final_xlsx) as xw:
        # 1) Per-Bild-Kombinationen
        if rows_combos:
            df_combos = pd.DataFrame(rows_combos).sort_values(["stem", "combo"])
            df_combos.to_excel(xw, index=False, sheet_name="per_image_combos")

            df_sum = df_combos.groupby("combo", as_index=False)["count"].sum().sort_values("combo")
            df_sum.to_excel(xw, index=False, sheet_name="summary_combos_total")

        # 2) Per-Bild Markerzusammenfassung
        if rows_marker_counts:
            df_markers = pd.DataFrame(rows_marker_counts).sort_values("stem")
            df_markers.to_excel(xw, index=False, sheet_name="per_image_marker_counts")

            cols_sum = ["total_cells", "CD45_pos", "CD45_neg", "Vim_pos", "Vim_neg", "CD52_pos", "CD52_neg"]
            df_marker_total = pd.DataFrame([df_markers[cols_sum].sum(numeric_only=True)])
            df_marker_total.to_excel(xw, index=False, sheet_name="summary_marker_total")

        # 3) QC pro Bild
        if rows_qc:
            df_qc = pd.DataFrame(rows_qc).sort_values("stem")
            df_qc.to_excel(xw, index=False, sheet_name="QC_per_image")

            # --- QC_summary (Aggregat) ---
            agg_cols = [c for c in df_qc.columns if c not in ("stem", "image_path")]
            df_qc_summary = pd.DataFrame({
                "metric": agg_cols,
                "mean":   [float(np.nanmean(df_qc[c])) for c in agg_cols],
                "median": [float(np.nanmedian(df_qc[c])) for c in agg_cols],
                "min":    [float(np.nanmin(df_qc[c])) for c in agg_cols],
                "max":    [float(np.nanmax(df_qc[c])) for c in agg_cols],
            })
            df_qc_summary.to_excel(xw, index=False, sheet_name="QC_summary")

            # --- Quick Review Tab ---
            # z-Scores (robust) für total_cells, all-marker-pos, und Laplacian-Varianz
            z_total, _, _ = robust_z(df_qc["total_cells"])
            z_allpos, _, _ = robust_z(df_qc["marker_all_pos_pct"])
            z_lap, _, _ = robust_z(df_qc["laplacian_var_dapi"])

            df_quick = df_qc[[
                "stem", "image_path", "total_cells",
                "multi_assign_pct_CD45", "multi_assign_pct_Vim", "multi_assign_pct_CD52",
                "CD45_fragile_pct", "Vim_fragile_pct", "CD52_fragile_pct",
                "marker_all_pos_pct",
                "laplacian_var_dapi",
                "blur_proxy_Vim", "blur_proxy_CD45", "blur_proxy_CD52"
            ]].copy()

            df_quick["z_total_cells"] = z_total
            df_quick["z_all_marker_pos"] = z_allpos
            df_quick["z_laplacian_var_dapi"] = z_lap

            # Red-Flags
            df_quick["flag_low_cells"] = df_quick["z_total_cells"] <= -3.0  # viel weniger DAPI als üblich
            df_quick["flag_multi_assign_1pct"] = (
                (df_quick["multi_assign_pct_CD45"] > MULTI_ASSIGN_LIMIT*100.0) |
                (df_quick["multi_assign_pct_Vim"]  > MULTI_ASSIGN_LIMIT*100.0) |
                (df_quick["multi_assign_pct_CD52"] > MULTI_ASSIGN_LIMIT*100.0)
            )
            df_quick["flag_high_fragile"] = (
                (df_quick["CD45_fragile_pct"] > 20.0) |
                (df_quick["Vim_fragile_pct"]  > 20.0) |
                (df_quick["CD52_fragile_pct"] > 20.0)
            )
            df_quick["flag_low_sharpness"] = df_quick["z_laplacian_var_dapi"] <= -3.0  # unscharf

            # Blur-Proxy kombinieren (größer = verdächtiger)
            df_quick["blur_proxy_total"] = df_quick[["blur_proxy_Vim","blur_proxy_CD45","blur_proxy_CD52"]].mean(axis=1)
            # Sortierung: stärkste Red-Flags zuerst
            df_quick.sort_values(
                ["flag_low_cells","flag_multi_assign_1pct","flag_low_sharpness","blur_proxy_total"],
                ascending=[False, False, False, False],
                inplace=True
            )

            df_quick.to_excel(xw, index=False, sheet_name="QC_quick_review")

        # 4) what-if Übersicht
        if whatif_all:
            df_whatif_all = pd.concat(whatif_all, ignore_index=True)
            df_whatif_all.to_excel(xw, index=False, sheet_name="QC_what_if_all")

    print(f"📊 Excel geschrieben: {final_xlsx}")


if __name__ == "__main__":
    main()


#dieses funktioniert PERFEKT

In [2]:
#analysis code!! this combines the geojson and calculates overlaps for Cxcl13

from pathlib import Path
from collections import Counter
import json
import math
import numpy as np  # wichtig für STRtree-Index-Rückgaben (np.int64)
import pandas as pd

from shapely.geometry import shape, mapping
from shapely.strtree import STRtree

# ---- Für Unschärfe-Score (DAPI aus OME-TIF laden) ----
from tifffile import imread as tif_imread
from skimage.filters import laplace

# -------------------- KONFIGURATION --------------------
# Oberordner, der die Bild-Ordner (C6_19, I7_18, …) enthält
ROOT = Path(r"C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cxcl13_images_analyzed")
# Zielordner für kombinierte Outputs & Excel
OUTPUT_ROOT = Path(r"C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cxcl13_images_analyzed")

# Marker-Namen (logisch) und erlaubte Ordner-Aliase (Suffix nach "<stem>__")
MARKER_ALIASES = {
    "DAPI": ["DAPI"],
    "CD45": ["CD45", "Cd45"], # case-tolerant
    "Vim":  ["VIM", "Vim"],   # case-tolerant
    "Cxcl13": ["Cxcl13", "CXCL13", "cxcl13"], # case-tolerant
}

# Schwellen (τ) für Coverage und DAPI-Puffer (Pixel) pro Marker
TAU = {
    "CD45": 0.90,
    "Vim":  0.92,
    "Cxcl13": 0.5,
}
DAPI_BUFFER_PX = {
    "CD45": 2,
    "Vim":  2,
    "Cxcl13": 2,
}

# Rettungsanker: DAPI-Centroid ∈ Marker → positiv (auch wenn Coverage knapp unter τ)
USE_CENTROID_BACKUP = True

# Sicherheitsgurt: max. Centroid-Distanz ≤ faktor * DAPI-Äquivalentradius (marker-spezifisch)
DIST_FACTOR = {
    "CD45": 1.5,
    "Vim":  1.5,
    "Cxcl13": 1.5,
}

# Mini-Gate: wenn echte Lokalität erkennbar ist, Distanzgurt nicht blockierend anwenden
# (Verhindert fälschliche Ablehnung bei großen zytoplasmatischen Masken)
TAU_GATE_MIN = 0.05  # 5% Mini-Overlap oder Centroid-in genügt, um Distanzgurt zu „übergehen“

# Idempotenz: Bild überspringen, wenn bereits combined.geojson existiert
SKIP_IF_COMBINED_EXISTS = True

# Wie der Anzeigename aussehen soll:
# - "positives_only": nur die Marker mit "+" (z. B. "CD45+ Vim+")
# - "full_combo":     vollständige Kombi (z. B. "DAPI+ CD45+ Vim- Cxcl13+")
NAME_STYLE = "positives_only"

# Marker-Polygone zusätzlich sichtbar mitschreiben?
INCLUDE_MARKER_GEOMETRIES = False  # bei Bedarf True setzen

# -------------------- BILDSUCHE / DAPI-KANAL (für Unschärfe) --------------------
# Gib hier 1..n Basisordner an, in denen die OME-TIFs liegen könnten.
# Wir suchen rekursiv nach Dateien, deren Name den STEM enthält und auf .ome.tif endet.
IMAGE_SEARCH_DIRS = [
    Path(r"D:\Image_analysis\QuPath_project_overall\export_sections"),
    # weitere Ordner nach Bedarf anhängen ...
]
# Wenn deine OME-TIFs konsistent sind, kannst du DAPI hier fix verdrahten.
# AUTO_ERKENNUNG versucht erst (C,Y,X) vs (Y,X,C) zu raten und nimmt Kanal 0 als DAPI.
# Optional: DAPI_CHANNEL_INDEX explizit setzen (z. B. 0).
AUTO_DETECT_CHANNEL_ORDER = True
DAPI_CHANNEL_INDEX = 0  # wird genutzt, wenn AUTO_DETECT_CHANNEL_ORDER True ist (wir nehmen dann Kanal 0)

# -------------------- QC-PARAMETER --------------------
# 1%-Grenze: maximal 1% DAPI-Zellen dürfen dieselbe Marker-Maske mit einer anderen DAPI-Zelle teilen
MULTI_ASSIGN_LIMIT = 0.01  # 1%


# -------------------- HILFSFUNKTIONEN --------------------
def read_geojson(path: Path):
    """GeoJSON laden → Listen von Geometrien und Properties."""
    data = json.loads(path.read_text(encoding="utf-8"))
    feats = data.get("features", [])
    geoms, props = [], []
    for f in feats:
        geoms.append(shape(f["geometry"]))
        props.append(f.get("properties", {}) or {})
    return geoms, props

def write_geojson(path: Path, features):
    """GeoJSON schreiben."""
    out = {"type": "FeatureCollection", "features": features}
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(out), encoding="utf-8")

def dapi_equiv_radius(area: float) -> float:
    """Äquivalentradius r für Kreis mit gleicher Fläche wie DAPI."""
    return math.sqrt(area / math.pi) if area > 0 else 0.0

def centroid_distance_ok(poly_dapi, poly_marker, factor: float):
    """
    Sicherheitsgurt prüfen:
    - Liefert (ok: bool, dist: float, r_equiv: float)
    - ok == True, wenn Distanz ≤ factor * r_equiv
    """
    r = dapi_equiv_radius(poly_dapi.area)
    if r <= 0:
        return True, 0.0, 0.0
    d = poly_dapi.centroid.distance(poly_marker.centroid)
    return (d <= factor * r), d, r

def coverage_fraction(dapi_poly, marker_poly, buffer_px=0):
    """Coverage = Fläche(DAPI_buffer ∩ Marker) / Fläche(DAPI_buffer)."""
    dapi_b = dapi_poly.buffer(buffer_px) if buffer_px and buffer_px > 0 else dapi_poly
    if not dapi_b.intersects(marker_poly):
        return 0.0
    inter = dapi_b.intersection(marker_poly)
    if inter.is_empty:
        return 0.0
    A = dapi_b.area
    return inter.area / A if A > 0 else 0.0

def build_index(geoms):
    """STRtree-Index + Rückmap von Geom-ID → Listenindex."""
    tree = STRtree(geoms)
    idmap = {id(g): i for i, g in enumerate(geoms)}
    return tree, idmap

def find_marker_folder(stem_dir: Path, stem: str, aliases):
    """Suche Unterordner <stem>__ALIAS (robust gegenüber Groß-/Kleinschreibung)."""
    for alias in aliases:
        cand = stem_dir / f"{stem}__{alias}"
        if cand.exists():
            return cand
    # Case-insensitive Fallback
    lower_aliases = [a.lower() for a in aliases]
    for sub in stem_dir.iterdir():
        if sub.is_dir() and sub.name.lower().startswith(f"{stem.lower()}__"):
            suffix = sub.name.split("__", 1)[1].lower()
            if suffix in lower_aliases:
                return sub
    return None

def find_filtered_geojson(marker_dir: Path, stem: str, alias: str):
    """Erwarte <stem>__<alias>_filtered.geojson im Marker-Ordner (mehrere Schreibweisen)."""
    candidates = [
        marker_dir / f"{stem}__{alias}_filtered.geojson",
        marker_dir / f"{stem}__{alias.upper()}_filtered.geojson",
        marker_dir / f"{stem}__{alias.capitalize()}_filtered.geojson",
    ]
    for p in candidates:
        if p.exists():
            return p
    # letzte Option: irgendein *_filtered.geojson im Ordner
    found = list(marker_dir.glob("*_filtered.geojson"))
    return found[0] if found else None

def combo_label(flags: dict):
    """Kombinationslabel aus Marker-Flags (DAPI ist per Definition positiv)."""
    parts = ["DAPI+"]
    parts.append("CD45+" if flags.get("CD45") else "CD45-")
    parts.append("Vim+"  if flags.get("Vim")  else "Vim-")
    parts.append("Cxcl13+" if flags.get("Cxcl13") else "Cxcl13-")
    return " ".join(parts)

def build_display_name(flags: dict, style: str = "positives_only"):
    """
    Erzeuge Anzeigename für die DAPI-Zelle.
    style: "positives_only" = nur positive Marker (z. B. "CD45+ Vim+")
           "full_combo"     = vollständige Kombi (z. B. "DAPI+ CD45- Vim+ Cxcl13-")
    """
    if style == "full_combo":
        return combo_label(flags)
    # positives_only
    pos = []
    for m in ["CD45", "Vim", "Cxcl13"]:
        if flags.get(m):
            pos.append(f"{m}+")
    return " ".join(pos) if pos else ""  # leer bei keiner Positivität


# -------------------- BILD-LADEN & UNSCHÄRFE (Laplacian-Varianz) --------------------
def find_image_file(stem: str):
    """Suche rekursiv nach einer .ome.tif, deren Dateiname den STEM enthält."""
    stem_lower = stem.lower()
    for root in IMAGE_SEARCH_DIRS:
        if not root.exists():
            continue
        # Suche breit: **/*<stem>*.ome.tif
        for p in root.rglob("*.ome.tif"):
            if stem_lower in p.name.lower():
                return p
    return None

def load_dapi_plane(image_path: Path):
    """
    Lädt den DAPI-Kanal als 2D-Array (float32 0..1).
    Heuristik: AUTO_DETECT_CHANNEL_ORDER → (C,Y,X) wenn Achse0 klein (<=6), sonst (Y,X,C).
    DAPI_CHANNEL_INDEX wird verwendet (Default 0).
    """
    arr = tif_imread(str(image_path))

    # Reduziere ggf. Z/T auf Einzelbild
    # Übliche OME-Layouts: (Y,X,C), (C,Y,X), (Z,Y,X,C), (T,Y,X,C), (T,Z,C,Y,X), ...
    # Wir ziehen die letzten 3 Achsen als (Y,X,C) oder (C,Y,X) heran.
    a = np.squeeze(arr)
    if a.ndim == 2:
        yx = a.astype(np.float32)
        return normalize01(yx)

    if a.ndim == 3:
        s0, s1, s2 = a.shape
        if AUTO_DETECT_CHANNEL_ORDER and s0 <= 6 and s0 < s1 and s0 < s2:
            # (C, Y, X)
            dapi = a[DAPI_CHANNEL_INDEX]
        else:
            # (Y, X, C)
            dapi = a[..., DAPI_CHANNEL_INDEX]
        return normalize01(dapi.astype(np.float32))

    # 4D oder 5D: versuche, auf (C,Y,X) oder (Y,X,C) zu projizieren
    # Wir nehmen die erste(n) Scheiben für T/Z.
    while a.ndim > 3:
        a = a[0]
    # nun 3D:
    s0, s1, s2 = a.shape
    if AUTO_DETECT_CHANNEL_ORDER and s0 <= 6 and s0 < s1 and s0 < s2:
        dapi = a[DAPI_CHANNEL_INDEX]
    else:
        dapi = a[..., DAPI_CHANNEL_INDEX]
    return normalize01(dapi.astype(np.float32))

def normalize01(img: np.ndarray):
    vmin = float(np.nanpercentile(img, 0.5))
    vmax = float(np.nanpercentile(img, 99.5))
    if vmax <= vmin:
        vmax = vmin + 1.0
    img = np.clip((img - vmin) / (vmax - vmin), 0.0, 1.0)
    return img.astype(np.float32)

def laplacian_variance_dapi(image_path: Path):
    """
    Berechnet die Laplacian-Varianz (Fokus/Schärfe) des DAPI-Kanals.
    Höher = schärfer. Gibt float oder np.nan bei Fehler zurück.
    """
    try:
        dapi = load_dapi_plane(image_path)
        lap = laplace(dapi)  # skimage Laplace
        var = float(np.var(lap))
        return var
    except Exception as e:
        print(f"⚠️ Laplacian-Varianz konnte nicht berechnet werden für {image_path}: {e}")
        return np.nan


# -------------------- QC-HELPER --------------------
def robust_z(x_vals):
    x = np.asarray(x_vals, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) or 1e-9
    return (x - med) / (1.4826 * mad), med, mad

def pct(numer, denom):
    return 0.0 if denom == 0 else 100.0 * float(numer) / float(denom)

def multi_assign_percent(marker_match_list, total_cells):
    # marker_match_list: Liste der Werte aus props[f"{m}_match"] (kann None enthalten)
    ids = [m for m in marker_match_list if m is not None]
    if total_cells == 0 or not ids:
        return 0.0
    c = Counter(ids)
    # Zellen, deren match-ID von >=2 Zellen benutzt wurde
    involved = sum(cnt for _, cnt in c.items() if cnt >= 2)
    return pct(involved, total_cells)

def fragile_positive_percent(rows, marker, tau):
    # rows: Iterable über Zell-Properties-Dicts
    pos = [r for r in rows if r.get(f"{marker}_pos", False)]
    if not pos:
        return 0.0
    fragile = [r for r in pos if (r.get(f"{marker}_coverage", 0.0) < tau and r.get(f"{marker}_centroid", False))]
    return pct(len(fragile), len(pos))

def dist_ratio_median(rows, marker):
    vals = []
    for r in rows:
        if r.get(f"{marker}_pos", False):
            d = r.get(f"{marker}_centroid_dist", None)
            req = r.get(f"{marker}_dapi_req_radius", None)
            if d is not None and req not in (None, 0.0):
                vals.append(d / req)
    return float(np.nanmedian(vals)) if vals else np.nan

def coverage_median(rows, marker):
    vals = [r.get(f"{marker}_coverage", np.nan) for r in rows if r.get(f"{marker}_pos", False)]
    return float(np.nanmedian(vals)) if vals else np.nan

def blur_proxy_score(rows, marker, tau):
    # 1) hoher Anteil Fragile-Positiva  2) niedriger Coverage-Median  3) hoher Distanz/Radius
    fp = fragile_positive_percent(rows, marker, tau) / 100.0          # 0..1
    cov_med = coverage_median(rows, marker)                            # 0..1
    cov_term = max(0.0, (tau - (cov_med if not np.isnan(cov_med) else tau)))
    dr = dist_ratio_median(rows, marker)                               # ~1 gut, >1.5 schlecht
    dr_term = 0.0 if np.isnan(dr) else max(0.0, (dr - 1.0)) / 0.5
    return float(0.5*fp + 0.3*cov_term + 0.2*dr_term)

def positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, tau, buffer_px, marker_name):
    geoms = marker_geoms.get(marker_name, [])
    tree, idmap = marker_idx.get(marker_name, (None, None))
    if not geoms or tree is None:
        return 0.0
    total = len(dapi_geoms)
    if total == 0:
        return 0.0
    pos = 0
    for poly_dapi in dapi_geoms:
        query_geom = poly_dapi.buffer(buffer_px) if buffer_px else poly_dapi
        best_cov = 0.0
        candidates = tree.query(query_geom)
        hit = False
        for cand in candidates:
            if isinstance(cand, (int, np.integer)):
                idx = int(cand); cand_geom = geoms[idx]
            else:
                cand_geom = cand
            cov = coverage_fraction(poly_dapi, cand_geom, buffer_px=buffer_px)
            centroid_hit = poly_dapi.centroid.within(cand_geom)
            if not centroid_hit and cov < TAU_GATE_MIN:
                ok_dist, _, _ = centroid_distance_ok(poly_dapi, cand_geom, DIST_FACTOR[marker_name])
                if not ok_dist:
                    continue
            if (cov >= tau) or (USE_CENTROID_BACKUP and centroid_hit):
                if cov > best_cov:
                    best_cov = cov
                    hit = True
        if hit:
            pos += 1
    return pct(pos, total)


# -------------------- KERNLOGIK --------------------
def classify_stem(stem_dir: Path):
    """
    Klassifiziert ein Bild (Ordner <stem_dir>):
    - Lädt DAPI + Marker (gefilterte GeoJSONs)
    - Wendet Coverage-τ + Centroid-Backup + Sicherheitsgurt an
    - Schreibt combined.geojson & combocounts.csv
    - Berechnet QC-Metriken & What-If & DAPI-Laplacian-Varianz
    - Gibt alle Strukturen zurück (für Aggregate & Big-Excel)
    """
    stem = stem_dir.name
    out_dir = OUTPUT_ROOT / stem
    combined_path = out_dir / f"{stem}__combined.geojson"
    counts_path   = out_dir / f"{stem}__combocounts.csv"
    per_image_xlsx = out_dir / f"{stem}__per_image_QC.xlsx"

    if SKIP_IF_COMBINED_EXISTS and combined_path.exists():
        print(f"♻️  Überschreibe bestehende Outputs für: {stem}")


    # --- DAPI laden ---
    dapi_dir = find_marker_folder(stem_dir, stem, MARKER_ALIASES["DAPI"])
    if dapi_dir is None:
        raise FileNotFoundError(f"DAPI-Ordner fehlt für {stem} unter {stem_dir}")
    dapi_gj = find_filtered_geojson(dapi_dir, stem, "DAPI")
    if dapi_gj is None or not dapi_gj.exists():
        raise FileNotFoundError(f"DAPI *_filtered.geojson fehlt in {dapi_dir}")
    dapi_geoms, dapi_props = read_geojson(dapi_gj)

    # --- Marker laden + indizieren ---
    marker_geoms = {}
    marker_props = {}
    marker_idx   = {}

    for logical_name, aliases in MARKER_ALIASES.items():
        if logical_name == "DAPI":
            continue
        mdir = find_marker_folder(stem_dir, stem, aliases)
        if mdir is None:
            print(f"⚠️  {logical_name}: Ordner fehlt für {stem}")
            marker_geoms[logical_name] = []
            marker_props[logical_name] = []
            marker_idx[logical_name] = (None, None)
            continue
        alias_hint = aliases[0]
        mgj = find_filtered_geojson(mdir, stem, alias_hint)
        if mgj is None or not mgj.exists():
            print(f"⚠️  {logical_name}: *_filtered.geojson fehlt in {mdir}")
            marker_geoms[logical_name] = []
            marker_props[logical_name] = []
            marker_idx[logical_name] = (None, None)
            continue
        g, p = read_geojson(mgj)
        marker_geoms[logical_name] = g
        marker_props[logical_name] = p
        marker_idx[logical_name]   = build_index(g) if g else (None, None)

    # Kurzer Diagnose-Print
    print(f"[{stem}] DAPI: {len(dapi_geoms)}  "
          f"CD45: {len(marker_geoms.get('CD45', []))}  "
          f"Vim: {len(marker_geoms.get('Vim', []))}  "
          f"Cxcl13: {len(marker_geoms.get('Cxcl13', []))}")

    # --- Klassifikation pro DAPI-Zelle ---
    out_features = []
    combos = Counter()
    pos_cd45 = pos_vim = pos_cxcl13 = 0
    total_cells = len(dapi_geoms)

    # Für QC: Listen der match-IDs je Marker
    match_ids = {"CD45": [], "Vim": [], "Cxcl13": []}

    for i, poly_dapi in enumerate(dapi_geoms):
        props = dict(dapi_props[i])
        props["DAPI"] = True

        flags = {}
        metrics = {}

        for m in ["CD45", "Vim", "Cxcl13"]:
            geoms = marker_geoms.get(m, [])
            tree, idmap = marker_idx.get(m, (None, None))
            tau = TAU[m]
            buf = DAPI_BUFFER_PX[m]
            dist_factor = DIST_FACTOR[m]

            best_cov = 0.0
            best_idx = None
            best_centroid_hit = False
            best_dist = None
            best_req_radius = None

            if geoms and tree is not None:
                query_geom = poly_dapi.buffer(buf) if buf else poly_dapi
                candidates = tree.query(query_geom)

                for cand in candidates:
                    if isinstance(cand, (int, np.integer)):
                        idx = int(cand)
                        cand_geom = geoms[idx]
                    else:
                        cand_geom = cand
                        idx = idmap[id(cand_geom)] if idmap is not None else geoms.index(cand_geom)

                    cov = coverage_fraction(poly_dapi, cand_geom, buffer_px=buf)
                    centroid_hit = poly_dapi.centroid.within(cand_geom)

                    if not centroid_hit and cov < TAU_GATE_MIN:
                        ok_dist, dcent, req_r = centroid_distance_ok(poly_dapi, cand_geom, dist_factor)
                        if not ok_dist:
                            continue
                    else:
                        ok_dist, dcent, req_r = centroid_distance_ok(poly_dapi, cand_geom, dist_factor)

                    if (cov >= tau) or (USE_CENTROID_BACKUP and centroid_hit):
                        if cov > best_cov:
                            best_cov = cov
                            best_idx = idx
                            best_centroid_hit = centroid_hit
                            best_dist = dcent
                            best_req_radius = req_r

            is_pos = best_idx is not None
            flags[m] = is_pos

            metrics[f"{m}_coverage"] = round(best_cov, 4)
            metrics[f"{m}_centroid"] = bool(best_centroid_hit)
            metrics[f"{m}_centroid_dist"] = None if best_dist is None else round(best_dist, 3)
            metrics[f"{m}_dapi_req_radius"] = None if best_req_radius is None else round(best_req_radius, 3)

            match_val = None
            if is_pos:
                base_props = marker_props[m][best_idx] if (m in marker_props and best_idx is not None and best_idx < len(marker_props[m])) else {}
                match_val = base_props.get("id") or base_props.get("label") or best_idx
            metrics[f"{m}_match"] = match_val
            match_ids[m].append(match_val)

        props.update(metrics)
        for m in ["CD45", "Vim", "Cxcl13"]:
            props[f"{m}_pos"] = bool(flags[m])

        combo = combo_label(flags)
        props["combo"] = combo
        props["name"]  = build_display_name(flags, NAME_STYLE)
        props["label"] = props["name"]

        out_features.append({
            "type": "Feature",
            "geometry": mapping(poly_dapi),
            "properties": props
        })
        combos[combo] += 1
        if flags.get("CD45"): pos_cd45 += 1
        if flags.get("Vim"):  pos_vim  += 1
        if flags.get("Cxcl13"): pos_cxcl13 += 1

    # Optional: Marker-Geometrien mitspeichern
    if INCLUDE_MARKER_GEOMETRIES:
        for m in ["CD45", "Vim", "Cxcl13"]:
            geoms = marker_geoms.get(m, [])
            props_list = marker_props.get(m, [])
            for gi, g in enumerate(geoms):
                basep = props_list[gi] if gi < len(props_list) else {}
                extra = {"layer": m, "name": m, "label": m}
                out_features.append({
                    "type": "Feature",
                    "geometry": mapping(g),
                    "properties": {**basep, **extra}
                })

    # --- Outputs pro Bild schreiben (GeoJSON/CSV) ---
    out_dir.mkdir(parents=True, exist_ok=True)
    write_geojson(combined_path, out_features)
    lines = ["combo,count"] + [f"{k},{v}" for k, v in sorted(combos.items())]
    counts_path.write_text("\n".join(lines), encoding="utf-8")

    print(f"✅ Geschrieben: {combined_path}")
    print(f"✅ Geschrieben: {counts_path}")

    marker_counts = {
        "stem": stem,
        "total_cells": total_cells,
        "CD45_pos": pos_cd45,
        "CD45_neg": total_cells - pos_cd45,
        "Vim_pos":  pos_vim,
        "Vim_neg":  total_cells - pos_vim,
        "Cxcl13_pos": pos_cxcl13,
        "Cxcl13_neg": total_cells - pos_cxcl13,
    }

    # -------------------- QC-METRIKEN (pro Bild) --------------------
    rows_props = [f["properties"] for f in out_features]  # nur DAPI-Features

    # Unschärfe-Score (Laplacian-Varianz) aus DAPI-Kanal
    img_path = find_image_file(stem)
    if img_path is None:
        print(f"⚠️ Kein OME-TIF zum STEM gefunden (für Unschärfe): {stem}")
        lap_var = np.nan
    else:
        lap_var = laplacian_variance_dapi(img_path)

    qc = {
        "stem": stem,
        "total_cells": total_cells,
        # Fragile-Positiva (%)
        "CD45_fragile_pct": fragile_positive_percent(rows_props, "CD45", TAU["CD45"]),
        "Vim_fragile_pct":  fragile_positive_percent(rows_props, "Vim",  TAU["Vim"]),
        "Cxcl13_fragile_pct": fragile_positive_percent(rows_props, "Cxcl13", TAU["Cxcl13"]),
        # Multi-Assign (% DAPI-Zellen involviert)
        "multi_assign_pct_CD45": multi_assign_percent(match_ids["CD45"], total_cells),
        "multi_assign_pct_Vim":  multi_assign_percent(match_ids["Vim"],  total_cells),
        "multi_assign_pct_Cxcl13": multi_assign_percent(match_ids["Cxcl13"], total_cells),
        # Distanz/Radius (Median)
        "centroid_dist_ratio_median_CD45": dist_ratio_median(rows_props, "CD45"),
        "centroid_dist_ratio_median_Vim":  dist_ratio_median(rows_props, "Vim"),
        "centroid_dist_ratio_median_Cxcl13": dist_ratio_median(rows_props, "Cxcl13"),
        # Coverage-Median (nur Positiva)
        "coverage_median_CD45": coverage_median(rows_props, "CD45"),
        "coverage_median_Vim":  coverage_median(rows_props, "Vim"),
        "coverage_median_Cxcl13": coverage_median(rows_props, "Cxcl13"),
        # Alle Marker positiv
        "marker_all_pos_pct": pct(sum(1 for r in rows_props if r.get("CD45_pos") and r.get("Vim_pos") and r.get("Cxcl13_pos")), total_cells),
        # Blur-Proxies
        "blur_proxy_Vim":  blur_proxy_score(rows_props, "Vim",  TAU["Vim"]),
        "blur_proxy_CD45": blur_proxy_score(rows_props, "CD45", TAU["CD45"]),
        "blur_proxy_Cxcl13": blur_proxy_score(rows_props, "Cxcl13", TAU["Cxcl13"]),
        # Reale Schärfemessung aus Bild
        "laplacian_var_dapi": lap_var,
        "image_path": str(img_path) if img_path else ""
    }
    # Flags (1%-Regel)
    qc["flag_multi_assign"] = any(qc[k] > (MULTI_ASSIGN_LIMIT*100.0) for k in (
        "multi_assign_pct_CD45", "multi_assign_pct_Vim", "multi_assign_pct_Cxcl13"
    ))

    # -------------------- WHAT-IF (pro Bild) --------------------
    whatif_rows = []
    for m in ["CD45", "Vim", "Cxcl13"]:
        base_tau = TAU[m]
        base_buf = DAPI_BUFFER_PX[m]
        recs = {
            "marker": m,
            f"PosRate_buf{base_buf-1 if base_buf>0 else 0}": positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, base_tau, max(0, base_buf-1), m),
            f"PosRate_buf{base_buf}":     positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, base_tau, base_buf, m),
            f"PosRate_buf{base_buf+1}":   positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, base_tau, base_buf+1, m),
            "PosRate_tau_minus0.02":      positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, max(0.0, base_tau-0.02), base_buf, m),
            "PosRate_tau_plus0.02":       positive_rate_with_params(dapi_geoms, marker_geoms, marker_idx, min(1.0, base_tau+0.02), base_buf, m),
        }
        whatif_rows.append(recs)
    df_whatif = pd.DataFrame(whatif_rows)

    # ---- per-Image-Excel (nur QC) schreiben ----
    with pd.ExcelWriter(per_image_xlsx) as xw:
        pd.DataFrame([qc]).to_excel(xw, index=False, sheet_name="QC_metrics")
        df_whatif.to_excel(xw, index=False, sheet_name="QC_what_if")
    print(f"🧪 QC-Excel geschrieben: {per_image_xlsx}")

    return {
        "stem": stem,
        "combos": combos,
        "marker_counts": marker_counts,
        "qc": qc,
        "whatif": df_whatif
    }


def main():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    stems = [p for p in ROOT.iterdir() if p.is_dir()]
    if not stems:
        print(f"⚠️ Keine Bild-Ordner unter {ROOT} gefunden")
        return

    print(f"🔎 {len(stems)} Bild-Ordner gefunden. Starte Verarbeitung…")

    rows_combos = []          # (stem, combo, count)
    rows_marker_counts = []   # per stem
    rows_qc = []              # per stem QC
    whatif_all = []           # pro stem (append marker-wise rows + stem)

    for stem_dir in stems:
        try:
            res = classify_stem(stem_dir)
            stem = res["stem"]
            combos = res["combos"]
            marker_counts = res["marker_counts"]
            qc = res["qc"]
            df_whatif = res["whatif"]

            for combo, count in combos.items():
                rows_combos.append({"stem": stem, "combo": combo, "count": count})
            if marker_counts:
                rows_marker_counts.append(marker_counts)
            if qc:
                rows_qc.append(qc)
            if isinstance(df_whatif, pd.DataFrame) and not df_whatif.empty:
                tmp = df_whatif.copy()
                tmp.insert(0, "stem", stem)
                whatif_all.append(tmp)

        except Exception as e:
            print(f"❌ {stem_dir.name}: {e}")

    if not rows_combos and not rows_marker_counts and not rows_qc:
        print("⚠️ Keine Daten zum Schreiben in Excel.")
        return

    final_xlsx = OUTPUT_ROOT / "__final_counts.xlsx"
    with pd.ExcelWriter(final_xlsx) as xw:
        # 1) Per-Bild-Kombinationen
        if rows_combos:
            df_combos = pd.DataFrame(rows_combos).sort_values(["stem", "combo"])
            df_combos.to_excel(xw, index=False, sheet_name="per_image_combos")

            df_sum = df_combos.groupby("combo", as_index=False)["count"].sum().sort_values("combo")
            df_sum.to_excel(xw, index=False, sheet_name="summary_combos_total")

        # 2) Per-Bild Markerzusammenfassung
        if rows_marker_counts:
            df_markers = pd.DataFrame(rows_marker_counts).sort_values("stem")
            df_markers.to_excel(xw, index=False, sheet_name="per_image_marker_counts")

            cols_sum = ["total_cells", "CD45_pos", "CD45_neg", "Vim_pos", "Vim_neg", "Cxcl13_pos", "Cxcl13_neg"]
            df_marker_total = pd.DataFrame([df_markers[cols_sum].sum(numeric_only=True)])
            df_marker_total.to_excel(xw, index=False, sheet_name="summary_marker_total")

        # 3) QC pro Bild
        if rows_qc:
            df_qc = pd.DataFrame(rows_qc).sort_values("stem")
            df_qc.to_excel(xw, index=False, sheet_name="QC_per_image")

            # --- QC_summary (Aggregat) ---
            agg_cols = [c for c in df_qc.columns if c not in ("stem", "image_path")]
            df_qc_summary = pd.DataFrame({
                "metric": agg_cols,
                "mean":   [float(np.nanmean(df_qc[c])) for c in agg_cols],
                "median": [float(np.nanmedian(df_qc[c])) for c in agg_cols],
                "min":    [float(np.nanmin(df_qc[c])) for c in agg_cols],
                "max":    [float(np.nanmax(df_qc[c])) for c in agg_cols],
            })
            df_qc_summary.to_excel(xw, index=False, sheet_name="QC_summary")

            # --- Quick Review Tab ---
            # z-Scores (robust) für total_cells, all-marker-pos, und Laplacian-Varianz
            z_total, _, _ = robust_z(df_qc["total_cells"])
            z_allpos, _, _ = robust_z(df_qc["marker_all_pos_pct"])
            z_lap, _, _ = robust_z(df_qc["laplacian_var_dapi"])

            df_quick = df_qc[[
                "stem", "image_path", "total_cells",
                "multi_assign_pct_CD45", "multi_assign_pct_Vim", "multi_assign_pct_Cxcl13",
                "CD45_fragile_pct", "Vim_fragile_pct", "Cxcl13_fragile_pct",
                "marker_all_pos_pct",
                "laplacian_var_dapi",
                "blur_proxy_Vim", "blur_proxy_CD45", "blur_proxy_Cxcl13"
            ]].copy()

            df_quick["z_total_cells"] = z_total
            df_quick["z_all_marker_pos"] = z_allpos
            df_quick["z_laplacian_var_dapi"] = z_lap

            # Red-Flags
            df_quick["flag_low_cells"] = df_quick["z_total_cells"] <= -3.0  # viel weniger DAPI als üblich
            df_quick["flag_multi_assign_1pct"] = (
                (df_quick["multi_assign_pct_CD45"] > MULTI_ASSIGN_LIMIT*100.0) |
                (df_quick["multi_assign_pct_Vim"]  > MULTI_ASSIGN_LIMIT*100.0) |
                (df_quick["multi_assign_pct_Cxcl13"] > MULTI_ASSIGN_LIMIT*100.0)
            )
            df_quick["flag_high_fragile"] = (
                (df_quick["CD45_fragile_pct"] > 20.0) |
                (df_quick["Vim_fragile_pct"]  > 20.0) |
                (df_quick["Cxcl13_fragile_pct"] > 20.0)
            )
            df_quick["flag_low_sharpness"] = df_quick["z_laplacian_var_dapi"] <= -3.0  # unscharf

            # Blur-Proxy kombinieren (größer = verdächtiger)
            df_quick["blur_proxy_total"] = df_quick[["blur_proxy_Vim","blur_proxy_CD45","blur_proxy_Cxcl13"]].mean(axis=1)
            # Sortierung: stärkste Red-Flags zuerst
            df_quick.sort_values(
                ["flag_low_cells","flag_multi_assign_1pct","flag_low_sharpness","blur_proxy_total"],
                ascending=[False, False, False, False],
                inplace=True
            )

            df_quick.to_excel(xw, index=False, sheet_name="QC_quick_review")

        # 4) what-if Übersicht
        if whatif_all:
            df_whatif_all = pd.concat(whatif_all, ignore_index=True)
            df_whatif_all.to_excel(xw, index=False, sheet_name="QC_what_if_all")

    print(f"📊 Excel geschrieben: {final_xlsx}")


if __name__ == "__main__":
    main()


#dieses funktioniert PERFEKT


🔎 34 Bild-Ordner gefunden. Starte Verarbeitung…
[C10_17] DAPI: 34793  CD45: 189  Vim: 699  Cxcl13: 15
✅ Geschrieben: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cxcl13_images_analyzed\C10_17\C10_17__combined.geojson
✅ Geschrieben: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cxcl13_images_analyzed\C10_17\C10_17__combocounts.csv
🧪 QC-Excel geschrieben: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cxcl13_images_analyzed\C10_17\C10_17__per_image_QC.xlsx
[C10_32] DAPI: 53903  CD45: 522  Vim: 1184  Cxcl13: 20
✅ Geschrieben: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - Strobel\Imaging\AnalysisV2\Full_analysis\Cxcl13_images_analyzed\C10_32\C10_32__combined.geojson
✅ Geschrieben: C:\Users\ostrobel\Indiana University\O365-IN-Phtx-Jerde Lab Main Storage - 